**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/mansvelder-2026"], check=True)
    os.chdir("/content/encore/entries/mansvelder-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Shamugia et al. (2026) on dandiset 000630, with dandisets 000636 and 000020

Paper: Shamugia, Weiler, ... Goriounova and Mansvelder (2026), "Ion Channels and Morphology Drive
Widespread Resonance in Human Cortical Interneurons", Research Square preprint
[doi:10.21203/rs.3.rs-11136357/v1](https://doi.org/10.21203/rs.3.rs-11136357/v1). The claims file
(`claims/mansvelder-2026.yaml`) was written from version 1 of the preprint and its supplement.

Dataset: [DANDI:000630](https://dandiarchive.org/dandiset/000630/0.230915.2257), "Human L1
patch-seq electrophysiology" (Chartrand et al. 2023), version **0.230915.2257**, the version the
paper cites (reference 27) and the only published version. It holds human layer 1 interneurons
(LAMP5/PAX6/Other and VIP), a part of the paper's human acute sample; the Chartrand table
puts 38 of its 210 cells in layers 2 to 4 (data check).

Outside inputs (protocol, Author, step 1.4), each pinned:

- [DANDI:000636](https://dandiarchive.org/dandiset/000636/0.241120.0510), "Human interneuron
  patch-seq electrophysiology" (Lee, Dalley et al. 2023, the paper's reference 2), version
  **0.241120.0510**, the only published version: human acute and slice-culture interneurons of all
  four subclasses. Labels per cell (subclass, acute or culture, donor, layer) from the Lee, Dalley
  et al. table (github.com/AllenInstitute/human_patchseq_gaba at commit 5eb357a), checked by the
  link claim `lee-label-link`. Four cells are in both 000630 and 000636; they are counted once.
- [DANDI:000020](https://dandiarchive.org/dandiset/000020/0.210913.1639), "Patch-seq recordings
  from mouse visual cortex" (Gouwens et al. 2020), version **0.210913.1639**, the only published
  version: the paper's mouse cells that are public, selected by the specimen ids of the paper's
  Table S2b through the Allen per-cell metadata (copy of 20200711_patchseq_metadata_mouse.csv at
  github.com/keon-arbabi/patch-seq-microglia, commit bb5c0e2), checked by `mouse-specimen-link`.
- Patch-seq expression of the human cells from the Chartrand et al. and Lee, Dalley et al.
  repositories at the same commits.
- Labels of the 000630 cells from the Chartrand et al. table (github.com/AllenInstitute/patchseq_human_L1
  at commit 7777857), checked by `chartrand-label-link`; the paper's Tables S1 (donor dataset,
  area, disease, lobe) and S2b (mouse specimen ids), read from its supplement.

All files are streamed; nothing is downloaded. The paper's mouse PV and SST cells are not in
000020 (they are among the 167 Table S2b ids not found), so no species comparison of PV or SST
cells can be run. No public source gives the pia-soma distance of more than 20 of the human cells
(22 rows of the Chartrand table), the ZD7288 recordings, or the paper's adapted model.

| Claim | Tested here |
|---|---|
| chartrand-label-link, lee-label-link, mouse-specimen-link (link claims added by this attempt) | tested |
| sample-size | tested (counts) |
| fig2b, fig2ab, fig2c, fig2d-subclass-proportion, fig2d-pv-highest, figs2a | tested |
| fig2e (two claims), fig2f, fig2g, fig3c, fig3d | tested on the subclasses both species have (LAMP5, VIP) |
| fig3eg, fig3ik | tested on human acute cells of four subclasses |
| fig3fh, fig3jl | tested on mouse LAMP5 and VIP cells |
| fig5g, fig5i, fig5jk, fig5l | tested on 000636 cultured cells against the human acute cells |
| figs1-region-pathology | tested |
| fig2d-sst-similar | blocked: the mouse SST cells are not public |
| fig4b, fig4c, fig4df, fig4g, figs2-l23, fig4i, fig4km | blocked: pia-soma distances public for 20 human cells only, and for no mouse cell |
| fig5be, figs3 | blocked: ZD7288 recordings not public |
| fig6b, fig6c | tested on the expression of the Chartrand et al. and Lee, Dalley et al. cells |
| figs4-culture-mouse-genes | tested on cultured cells; the mouse part is not tested |
| fig6a | blocked: Allen Brain Map data not analysed in the time of these runs |
| fig7, figs5 and fig8 model claims (eight) | blocked: models not rebuilt (see the attempt record) |

In [ ]:
import time, resource, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import resonance as R

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
SOURCES = {"000630": (R.DANDISET, R.VERSION), "000636": (R.LEE_DANDISET, R.LEE_VERSION),
           "000020": (R.MOUSE_DANDISET, R.MOUSE_VERSION)}
print({k: v[1] for k, v in SOURCES.items()})

## Loading

Every file of dandiset 000630 is read, with the files of 000636 that the Lee, Dalley et al.
table lists and the files of 000020 whose cell is one of the paper's mouse cells (Table S2b
specimen ids, mapped to 000020 sessions by the Allen metadata). Each file is opened once; its
chirp and hyperpolarizing long-square sweeps are reduced as they are read (see `resonance.py`).

In [ ]:
t0 = time.time()
lee = R.lee_table()
mtab = R.mouse_table()
assets = {k: R.list_assets(*v) for k, v in SOURCES.items()}
s2b_sessions = set(mtab.loc[mtab["cell_specimen_id"].isin(list(R.TABLE_S2B)), "session_id"])
use = {"000630": assets["000630"],
       "000636": [a for a in assets["000636"] if R.session_of(a["path"]) in set(lee["session_id"])],
       "000020": [a for a in assets["000020"] if R.session_of(a["path"]) in s2b_sessions]}
print({k: f"{len(use[k])} of {len(assets[k])} files" for k in use})
files = {k: R.process_files(v, workers=12, all_cells=True) for k, v in use.items()}
print(f"read in {time.time() - t0:.0f} s; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
SUB_HUMAN = {"LAMP5/PAX6/Other": "LAMP5", "VIP": "VIP", "PVALB": "PV", "SST": "SST"}
SUB_MOUSE = {"Lamp5": "LAMP5", "Vip": "VIP", "Sncg": "VIP", "Pvalb": "PV", "Sst": "SST"}
CRE_MOUSE = {"Ndnf": "LAMP5", "Lamp5": "LAMP5", "Vip": "VIP", "Pvalb": "PV", "Sst": "SST"}

frames = {k: pd.DataFrame([R.measures(r) for r in v]).assign(source=k) for k, v in files.items()}
# 000630, labels from the Chartrand et al. table
ct = R.chartrand_table()
a = frames["000630"].merge(ct[["session_id", "spec_id", "donor_name", "subclass", "t-type", "structure",
                               "medical_conditions", "donor_age", "donor_sex", "Depth_from_Pia_microns", "target_layer", "roi"]],
                           on="session_id", how="left")
a["donor"] = a["donor_name"]
a["sub"] = a["subclass"].map(SUB_HUMAN)
a["prep"] = "acute"
a["species"] = "human"
# layer as the Chartrand table gives it (target_layer); the dandiset describes its cells as layer 1 cells
a["layer"] = a["target_layer"].map(lambda x: str(int(x)) if pd.notna(x) else "none")
a["spec"] = pd.to_numeric(a["spec_id"], errors="coerce").astype("Int64").astype(str)
# 000636, labels from the Lee, Dalley et al. table
b = frames["000636"].merge(lee[["session_id", "specimen_id_x", "Donor", "Revised_subclass_label", "subclass_label",
                                "Transcriptomic_type", "condition", "Cortical_layer", "structure", "lobe",
                                "disease_category"]], on="session_id", how="left")
b["donor"] = b["Donor"].str.upper()
b["sub"] = b["Revised_subclass_label"].map(SUB_HUMAN)
b["prep"] = b["condition"]
b["species"] = "human"
b["layer"] = b["Cortical_layer"].astype(str)
b["spec"] = pd.to_numeric(b["specimen_id_x"], errors="coerce").astype("Int64").astype(str)
# 000020, labels from the Allen metadata; the paper's mouse number from Table S2b
m = frames["000020"].merge(mtab[["session_id", "cell_specimen_id", "cell_specimen_name", "donor_name", "biological_sex",
                                 "full_genotype", "structure", "T-type Label", "cell_soma_normalized_depth",
                                 "depth_from_pia_um"]], on="session_id", how="left")
m["s2b_mouse"] = m["cell_specimen_id"].map(R.TABLE_S2B)
m["donor"] = m["s2b_mouse"].astype(str)
ttype_sub = m["T-type Label"].str.split().str[0]
cre_sub = m["cell_specimen_name"].str.extract(r"^([A-Za-z0-9]+)")[0].map(CRE_MOUSE)
m["sub"] = ttype_sub.map(SUB_MOUSE).fillna(cre_sub)                       # reading a: Sncg as VIP
m["sub_sncg_lamp5"] = ttype_sub.map({**SUB_MOUSE, "Sncg": "LAMP5"}).fillna(cre_sub)   # reading b
m["sub_source"] = np.where(ttype_sub.notna(), "T-type", np.where(cre_sub.notna(), "Cre line", "none"))
m["prep"] = "acute"
m["species"] = "mouse"
m["layer"] = m["structure"].str.replace("VISp", "", regex=False)

# Cells deposited in both 000630 and 000636 are counted once, from 000630
dup = sorted(set(a["session_id"]) & set(b["session_id"]))
b_dup = b[b["session_id"].isin(dup)]
b = b[~b["session_id"].isin(dup)]
cols = ["source", "spec", "session_id", "subject", "donor", "species", "prep", "sub", "layer", "n_chirp", "fres_a", "Q_a",
        "f3db_a", "fres_b", "Q_b", "f3db_b", "zmax_b", "z0_a", "z0_b", "vm_a", "vm_b", "sag_a", "sag_b", "rin_a", "rin_b"]
allc = pd.concat([a.reindex(columns=cols + ["Depth_from_Pia_microns"]), b.reindex(columns=cols),
                  m.reindex(columns=cols + ["sub_sncg_lamp5", "cell_soma_normalized_depth"])], ignore_index=True)
allc["chirp_ok"] = allc["n_chirp"].fillna(0) > 0
allc["sag_ok"] = allc["sag_a"].notna()
hum = allc[allc["species"] == "human"]
acute = allc[(allc["species"] == "human") & (allc["prep"] == "acute")]
cult = allc[(allc["species"] == "human") & (allc["prep"] == "culture")]
mouse = allc[allc["species"] == "mouse"]
print("cells with any measure:", allc.groupby(["species", "prep", "source"]).size().to_dict())
print("duplicate sessions in 000630 and 000636 (counted once):", len(dup))

## Data check

In [ ]:
for k, (ds, ver) in SOURCES.items():
    fr = frames[k]
    print(f"{k} ({ver}): {len(fr)} files read of {len(assets[k])}; subjects {fr['subject'].nunique()}; "
          f"species {fr['nwb_species'].value_counts().to_dict()}")
    stim = {}
    for r in files[k]:
        for s_ in {x["stimulus"].rsplit("_DA", 1)[0] for x in r["sweeps"]}:
            if "CHIRP" in s_ or any(t in s_ for t in R.LS_NAMES):
                stim[s_] = stim.get(s_, 0) + 1
    print("   files holding each chirp or long-square stimulus:", dict(sorted(stim.items())))
    print("   files with a chirp sweep:", int(fr["has_chirp"].sum()), "| with a usable 20 s chirp:",
          int((fr["n_chirp"].fillna(0) > 0).sum()), "| with a sag ratio:", int(fr["sag_a"].notna().sum()))
    drops = fr[["chirp_not20s", "chirp_truncated", "chirp_lead_zero_only", "chirp_spike", "ls_truncated"]].fillna(0).sum()
    print("   dropped sweeps:", drops.astype(int).to_dict(),
          "| files whose chirp sweeps were all dropped:", int((fr["has_chirp"] & (fr["n_chirp"].fillna(0) == 0)).sum()))
print("\nsweeps dropped per file, files with any dropped chirp or long square:")
dr = pd.concat(frames.values())
dr = dr[dr[["chirp_not20s", "chirp_truncated", "chirp_spike", "ls_truncated"]].fillna(0).sum(axis=1) > 0]
print(dr.groupby("source")[["chirp_not20s", "chirp_truncated", "chirp_lead_zero_only", "chirp_spike", "ls_truncated", "n_chirp"]]
      .apply(lambda g: g.fillna(0).astype(int).value_counts().head(8)).to_string())
c = allc[allc["chirp_ok"]]
print("\nranges over cells with a usable chirp, by group:")
for v, u in [("fres_b", "Hz"), ("Q_b", ""), ("f3db_b", "Hz"), ("fres_a", "Hz"), ("Q_a", ""), ("z0_b", "MOhm"), ("vm_a", "mV")]:
    print(f"  {v:7s}", {g: f"{x[v].min():.3f} to {x[v].max():.3f} (median {x[v].median():.3f}, NaN {x[v].isna().sum()})"
                         for g, x in c.groupby(["species", "prep"])}, u)
s_ = allc[allc["sag_ok"]]
for v, u in [("sag_a", ""), ("sag_b", ""), ("rin_a", "MOhm"), ("rin_b", "MOhm"), ("vm_b", "mV")]:
    print(f"  {v:7s}", {g: f"{x[v].min():.3f} to {x[v].max():.3f} (median {x[v].median():.3f})"
                         for g, x in s_.groupby(["species", "prep"])}, u)
print("chirp window start (s):", sorted(pd.concat(frames.values())["chirp_start"].dropna().round(2).unique()))
print("long squares whose minimum lies below the pre-step baseline:",
      {k: f"{int(fr['ls_deflect_ok'].sum())} of {int(fr['ls_n'].sum())}" for k, fr in frames.items()})
# recordings deposited twice: identical measures between cells
key = allc[allc["chirp_ok"]][["z0_a", "fres_a", "rin_a", "sag_a"]].round(6).astype(str).agg("|".join, axis=1)
print("chirp cells sharing identical impedance, sag and input resistance with another cell:", int(key.duplicated(keep=False).sum()))
bd = b_dup.set_index("session_id")[["fres_b", "Q_b", "sag_a", "rin_a"]]
ad = a.set_index("session_id").loc[bd.index, ["fres_b", "Q_b", "sag_a", "rin_a"]]
print("sessions in both 000630 and 000636:", len(dup), "| identical measures in both copies:",
      int(np.isclose(ad.fillna(-1).to_numpy(), bd.fillna(-1).to_numpy()).all(axis=1).sum()))
print("human donors in both 000630 and 000636 (by donor name):",
      len(set(a["donor"].dropna()) & set(b["donor"].dropna())))
print("mouse: pia-soma distance filled for", int(m["depth_from_pia_um"].notna().sum()), "cells; normalized soma depth for",
      int(m["cell_soma_normalized_depth"].notna().sum()))
dep = lambda cols: [c for c in cols if any(w in c.lower() for w in ("depth", "pia", "dist"))]
print("label table columns naming depth, pia or distance: Lee, Dalley et al.", dep(lee.columns), "| Chartrand et al.", dep(ct.columns),
      "| Allen mouse metadata", dep(mtab.columns))
print("Lee, Dalley et al. table, all columns:", list(lee.columns))
print("subject fields in the files (general/subject):", {k: sorted(set().union(*[r["subject"].keys() for r in v])) for k, v in files.items()})
print("000630 cells by the Chartrand table's target_layer:", a["layer"].value_counts().to_dict(), "| roi:", a["roi"].value_counts().to_dict())

## chartrand-label-link

Link claim (added by this attempt). The subclass and donor labels of the 000630 cells come
from the Chartrand et al. (2023) table, matched by the session id of each NWB file (the
"Ephys_Roi_Result" id in the table's `nwb_path`). The link holds when every file matches one
row, each NWB subject id maps to one donor name and back, and sex and age agree.

In [ ]:
matched = a["spec_id"].notna()
print("files matched to a row of the Chartrand table:", int(matched.sum()), "of", len(a))
print("NWB subjects with more than one donor name:", int((a.groupby("subject")["donor_name"].nunique() > 1).sum()),
      "| donor names with more than one NWB subject:", int((a.groupby("donor_name")["subject"].nunique() > 1).sum()))
print("sex, files against table:")
print(pd.crosstab(a["nwb_sex"], a["donor_sex"]))
age_days = a["nwb_age"].str.extract(r"P([\d.]+)D")[0].astype(float)
age_years = a["donor_age"].astype(str).str.extract(r"(\d+)")[0].astype(float)
diff = (age_days / 365.25 - age_years)
print(f"age, files (days / 365.25) minus table (years): {diff.min():.2f} to {diff.max():.2f} over {diff.notna().sum()} files")
print("subclass of chirp cells (table):", a.loc[a["n_chirp"].fillna(0) > 0, "subclass"].value_counts().to_dict())
donors630 = sorted(a["donor_name"].dropna().unique())
where = {d: [k for k, v in R.TABLE_S1_DONORS.items() if d in v] for d in donors630}
print("000630 donors in Table S1:", sum(bool(w) for w in where.values()), "of", len(donors630),
      "| their Table S1 dataset:", pd.Series([",".join(w) or "absent" for w in where.values()]).value_counts().to_dict())
ch = a[a["n_chirp"].fillna(0) > 0]
print("donors of chirp cells:", ch["donor_name"].nunique(), "| their Table S1 dataset:",
      pd.Series([",".join(where[d]) or "absent" for d in ch["donor_name"].dropna().unique()]).value_counts().to_dict())
print("t-types of LAMP5/PAX6/Other chirp cells naming SST:",
      ch.loc[(ch["subclass"] == "LAMP5/PAX6/Other") & ch["t-type"].str.contains("SST", na=False), "t-type"].value_counts().to_dict())

**Result: reproduced.** All 210 files of 000630 match one row of the Chartrand table by session id; subject ids and donor names map one to one; sex agrees for 209 files (one file gives "unknown"), and age differs by -0.06 to -0.01 years. All 53 donors are in Table S1, 43 under "AcuteLDPublished" and 10 under "Chartrand"; of the 31 donors of chirp cells, 28 and 3. The 85 chirp cells are LAMP5/PAX6/Other 67 (3 of them with the t-type SST CHRNA4) and VIP 18.

## lee-label-link

Link claim (added by this attempt). The subclass, preparation (acute or culture) and donor of
each 000636 cell come from the per-cell table of Lee, Dalley et al. (2023)
(github.com/AllenInstitute/human_patchseq_gaba at commit 5eb357a), matched by session id (its
`nwb_id`). The link holds when at least 90% of the files match one row, each NWB subject id
maps to one donor name and back, and the donors' datasets in Table S1 agree with the
preparation (acute cells from "AcuteLDPublished" donors, cultured cells from
"PublishedLD_Cultured" donors or donors with acute cells).

In [ ]:
fb = frames["000636"].merge(lee[["session_id", "Donor", "condition", "Revised_subclass_label"]], on="session_id", how="left")
fb["donor"] = fb["Donor"].str.upper()
print("000636 files:", len(assets["000636"]), "| matched to one row of the Lee table:", int(fb["Donor"].notna().sum()),
      f"({100 * fb['Donor'].notna().sum() / len(assets['000636']):.1f}%)",
      "| table rows with a session id:", len(lee), "| rows matching no file:", int((~lee["session_id"].isin(fb["session_id"])).sum()))
print("rows per matched session:", lee[lee["session_id"].isin(fb["session_id"])].groupby("session_id").size().max())
print("NWB subjects with more than one donor name:", int((fb.groupby("subject")["donor"].nunique() > 1).sum()),
      "| donor names with more than one NWB subject:", int((fb.groupby("donor")["subject"].nunique() > 1).sum()))
fb["s1"] = fb["donor"].map(lambda d: R.TABLE_S1.get(d, ("absent",))[0])
print("Table S1 dataset of each cell's donor, by preparation:")
print(pd.crosstab(fb["s1"], fb["condition"]))
print("cells by subclass and preparation:")
print(pd.crosstab(fb["Revised_subclass_label"], fb["condition"]))
ages = fb["nwb_age"].str.extract(r"P([\d.]+)D")[0].astype(float) / 365.25
print(f"NWB donor ages: {ages.min():.1f} to {ages.max():.1f} years; NWB sex {fb['nwb_sex'].value_counts().to_dict()}")
print("files with an NWB age of P0.0D:", int((ages == 0).sum()), "from", fb.loc[ages == 0, "subject"].nunique(), "subjects (donors",
      sorted(fb.loc[ages == 0, "donor"].dropna().unique()), ")")

**Result: reproduced.** 690 of the 706 files of 000636 (97.7%) match one row of the Lee, Dalley et al. table by session id; the other 16 have no session id in the table (44 table rows have none). Subject ids and donor names map one to one. Every acute cell's donor is listed in Table S1 under "AcuteLDPublished"; cultured cells come from donors listed under "PublishedLD_Cultured" (272), "AcuteLDPublished" (147), "Chartrand" (17) or not listed (20, one donor, H19.03.002). The table gives 234 acute cells (LAMP5 40, PV 115, SST 19, VIP 60) and 456 cultured cells with a file. 21 files, from 6 subjects, give the donor's age as P0.0D (printed above).

## mouse-specimen-link

Link claim (added by this attempt). The paper lists its mouse cells by Allen cell specimen id
(Table S2b). The Allen metadata of the mouse Patch-seq release (Gouwens et al. 2020) maps each
specimen id to the ephys session id of a 000020 file and gives its T-type. The link holds when
each matched cell's Allen name carries the mouse number Table S2b gives it, and the file's
subject sex and genotype agree with the Allen metadata; the fraction of Table S2b found is
reported (partially reproduced when it is below 90%).

In [ ]:
found = mtab["cell_specimen_id"].isin(list(R.TABLE_S2B))
print("Table S2b specimen ids:", len(R.TABLE_S2B), "| in the Allen metadata (and so in 000020):", int(found.sum()),
      f"({100 * found.sum() / len(R.TABLE_S2B):.1f}%) | 000020 files read for them:", len(m))
ok_mouse = m.apply(lambda r: f"-{r['s2b_mouse']}." in str(r["cell_specimen_name"]), axis=1)
print("Allen cell name carries the Table S2b mouse number:", int(ok_mouse.sum()), "of", len(m))
print("file sex agrees with Allen sex:", int((m["nwb_sex"] == m["biological_sex"]).sum()), "of", len(m))
gt = m.apply(lambda r: str(r["nwb_genotype"]).split("/")[0].split(";")[0] in str(r["full_genotype"]), axis=1)
print("file genotype's first allele found in the Allen genotype:", int(gt.sum()), "of", len(m))
print("one 000020 subject per Table S2b mouse:", int((m.groupby("s2b_mouse")["subject"].nunique() == 1).sum()), "of",
      m["s2b_mouse"].nunique(), "mice")
miss = pd.Series([k for k in R.TABLE_S2B if k not in set(mtab["cell_specimen_id"])])
print("Table S2b ids not found, by specimen id range:", pd.cut(miss, [0, 9e8, 1.1e9, 1.2e9, 2e9]).value_counts().sort_index().to_dict())
print("subclass of the matched cells (T-type, else Cre line):", m["sub"].value_counts(dropna=False).to_dict(),
      "| source:", m["sub_source"].value_counts().to_dict())
print("T-type subclass words:", m["T-type Label"].str.split().str[0].value_counts(dropna=False).to_dict())
print("paper's mouse counts for sag (Table S3, Fig. 3c): LAMP5 435, VIP 101, PV 39, SST 44")

**Result: partially reproduced.** 461 of the 628 Table S2b specimen ids (73.4%) are in the Allen metadata and so in 000020. For all 461, the Allen cell name carries the mouse number Table S2b gives, the file's sex agrees with the Allen sex, the file genotype agrees with the Allen genotype, and each Table S2b mouse is one 000020 subject. The matched cells are LAMP5 374 and VIP 86 (Sncg T-types counted as VIP) and no PV or SST cells, against the paper's LAMP5 435, VIP 101, PV 39 and SST 44, so the paper's mouse PV and SST cells are among the 167 ids not found (90 of them above 1.2e9).

## sample-size

> The dataset comprises 1085 human interneurons from 259 subjects (542 acute from 170 donors,
> 543 cultured from 89 donors) and 619 acutely recorded mouse interneurons from 293 mice.

The criterion counts the human acute cells and donors the attempt assembles, against 542 and
170. Here these are the 000630 cells with the acute cells of 000636 (cells in both counted once).

In [ ]:
print("human acute interneurons:", len(acute), "(000630", int((acute.source == "000630").sum()), "+ 000636",
      int((acute.source == "000636").sum()), "; expected 542 +/- 54.2)")
print("their donors:", acute["donor"].nunique(), "(expected 170 +/- 17)")
print("000630 alone:", len(a), "cells from", a["subject"].nunique(), "donors")
print("human cultured interneurons:", len(cult), "from", cult["donor"].nunique(), "donors (paper 543 from 89)")
print("mouse interneurons:", len(mouse), "from", mouse["donor"].nunique(), "mice (paper 619 from 293; Table S2b lists",
      len(R.TABLE_S2B), "cells from", len(set(R.TABLE_S2B.values())), "mice)")
for nm, d in [("human acute", acute), ("human cultured", cult), ("mouse", mouse)]:
    c = d[d.chirp_ok]
    print(f"{nm} with a usable chirp: {len(c)}", c["sub"].value_counts().to_dict())
print("paper chirp counts: human acute 297 (LAMP5 144, VIP 60, PV 78, SST 15); mouse 248 (LAMP5 120, VIP 49, PV 36, SST 43);"
      " cultured 268 (LAMP5 21, VIP 28, PV 125, SST 94)")

**Result: not reproduced.** The data assembled hold 440 human acute interneurons (210 from 000630, 230 from 000636, 4 cells deposited in both counted once) from 76 donors, against 542 +/- 54.2 from 170 +/- 17. 212 of them have a usable chirp (paper 297). There are 456 cultured cells from 54 donors (paper 543 from 89) and 461 mouse cells from 221 mice (paper 619 from 293).

## Chirp analysis: which reading follows the paper

fres, Q and f3dB per cell under the two readings of O1: frequencies kept from 0.2 Hz (IPFX's
default, reading a) or from 0.4 Hz (the stimulus band, reading b). Table S3 gives medians per
group; several are exactly 0.40 Hz (human SST, mouse VIP, mouse SST), which is the value a
profile without a peak takes when the lowest frequency kept is 0.4 Hz, and which reading (a)
cannot give. The medians and resonant fractions under each reading are set against Table S3
here. Cells whose peak is the lowest frequency kept have Q = 1 and fres at the floor: fres is
censored there, which the tests of fres below inherit (they are rank tests or linear models,
as in the paper).

In [ ]:
T3 = {("human", "LAMP5"): dict(fres=1.09, Q=1.013, f3db=7.36, frac=31.25), ("human", "VIP"): dict(fres=0.95, frac=25.0),
      ("human", "PV"): dict(fres=2.18, Q=1.04, f3db=13.03, frac=43.6), ("human", "SST"): dict(fres=0.40, frac=20.0),
      ("mouse", "LAMP5"): dict(fres=1.15, f3db=15.20), ("mouse", "VIP"): dict(fres=0.40), ("mouse", "PV"): dict(fres=1.30, Q=1.008, f3db=20.86),
      ("mouse", "SST"): dict(fres=0.40)}
ac = allc[allc["chirp_ok"] & (allc["prep"] == "acute") & allc["sub"].notna()]
rows = []
for (sp, sb), g in ac.groupby(["species", "sub"]):
    row = dict(species=sp, sub=sb, n=len(g))
    for rd in "ab":
        res = R.is_resonant(g[f"fres_{rd}"], g[f"Q_{rd}"])
        row.update({f"fres_{rd}": round(g[f"fres_{rd}"].median(), 3), f"Q_{rd}": round(g[f"Q_{rd}"].median(), 4),
                    f"f3db_{rd}": round(g[f"f3db_{rd}"].median(), 2), f"res%_{rd}": round(100 * res.mean(), 1),
                    f"floor_{rd}": int((g[f"Q_{rd}"] == 1).sum()),
                    f"fres>1,Q<=1.05_{rd}": int(((g[f"fres_{rd}"] > 1) & (g[f"Q_{rd}"] <= 1.05)).sum())})
    row.update({f"S3 {k}": v for k, v in T3.get((sp, sb), {}).items()})
    rows.append(row)
print(pd.DataFrame(rows).to_string())
print("\nfres histogram (counts per bin, Hz), reading b:")
bins = [0, 0.41, 1, 1.5, 2, 3, 4, 5, 8]
for (sp, pr), g in allc[allc["chirp_ok"]].groupby(["species", "prep"]):
    print(f"  {sp} {pr}:", pd.cut(g["fres_b"], bins).value_counts().sort_index().values.tolist(), "bins", bins)
print("Q histogram, reading b, bins 1, 1.02, 1.05, 1.1, 1.2, 1.5, inf:")
for (sp, pr), g in allc[allc["chirp_ok"]].groupby(["species", "prep"]):
    print(f"  {sp} {pr}:", pd.cut(g["Q_b"], [0.99, 1.0, 1.02, 1.05, 1.1, 1.2, 1.5, np.inf]).value_counts().sort_index().values.tolist())
# The Allen Institute's own IPFX chirp features for the 000636 cells (LeeDalley_ephys_fx.csv in
# the same pinned repository) as a check of the helper's impedance profile; not used in any test.
fx = pd.read_csv(R.LEE_TABLE.replace("LeeDalley_manuscript_metadata_v2.csv", "LeeDalley_ephys_fx.csv"))
chk = b.merge(fx[["specimen_id", "peak_freq_chirp", "peak_ratio_chirp", "3db_freq_chirp"]],
              left_on="specimen_id_x", right_on="specimen_id", how="inner")
chk = chk[(chk["n_chirp"].fillna(0) > 0) & chk["peak_freq_chirp"].notna()]
print(f"\n000636 cells with Allen chirp features and a usable chirp here: {len(chk)}")
for mine, theirs in [("fres_a", "peak_freq_chirp"), ("fres_b", "peak_freq_chirp"), ("Q_a", "peak_ratio_chirp"),
                     ("Q_b", "peak_ratio_chirp"), ("f3db_b", "3db_freq_chirp")]:
    print(f"  {mine} against Allen {theirs}: Spearman rho {stats.spearmanr(chk[mine], chk[theirs], nan_policy="omit")[0]:.3f}; "
          f"medians {chk[mine].median():.3f} and {chk[theirs].median():.3f}")
print("  Allen peak frequency, lowest values:", sorted(chk["peak_freq_chirp"].round(2).unique())[:5],
      "| fraction of cells with Allen fres > 1 and peak ratio > 1.05:",
      round(float(((chk['peak_freq_chirp'] > 1) & (chk['peak_ratio_chirp'] > 1.05)).mean()), 3),
      "| here, reading b:", round(float(R.is_resonant(chk['fres_b'], chk['Q_b']).mean()), 3))

The LOWESS fraction (0.04, which the Methods give for the model chirp, O1) spans about 1.6 Hz
on the linear frequency axis, so it flattens slow, weak peaks and lowers Q. How the medians and
resonant fractions move with the fraction (reading b of O1; 0.04 is the one used):

In [ ]:
chirps_of = {(k, r["session_id"]): r["cell"]["chirps"] for k, v in files.items() for r in v if r["cell"] and r["cell"]["chirps"]}
sm_rows, smooth_vals = [], {}
for frac in (0.02, 0.04, 0.08):
    vals = {key: R.resonance(*[x for i, x in enumerate(R.impedance(ch, "b", frac)) if i != 1]) for key, ch in chirps_of.items()}
    smooth_vals[frac] = vals
    t = allc[allc["chirp_ok"] & allc["sub"].notna()].copy()
    t["fr"] = [vals[(s_, x)]["fres"] for s_, x in zip(t["source"], t["session_id"])]
    t["q"] = [vals[(s_, x)]["Q"] for s_, x in zip(t["source"], t["session_id"])]
    for (sp, pr, sb), g in t.groupby(["species", "prep", "sub"]):
        sm_rows.append(dict(frac=frac, group=f"{sp} {pr} {sb}", n=len(g), med_fres=round(g["fr"].median(), 2), med_Q=round(g["q"].median(), 4),
                            resonant_pct=round(100 * R.is_resonant(g["fr"], g["q"]).mean(), 1)))
print(pd.DataFrame(sm_rows).pivot(index="group", columns="frac", values=["med_fres", "med_Q", "resonant_pct"]).to_string())

Reading (b) of O1 is the main reading from here on. Three things support it: Table S3's
medians of exactly 0.40 Hz, which only a 0.4 Hz floor gives; the mouse medians, which reading
(b) matches (LAMP5 fres 1.15 Hz and f3dB 15.2 Hz against Table S3's 1.15 and 15.20) and
reading (a) does not (0.20 Hz); and the Allen Institute's own chirp features for the 000636
cells, whose lowest peak frequency is 0.4 Hz and whose medians and resonant fraction are close
to reading (b)'s (printed above). Reading (a) is shown for every claim. Neither reading gives
the human LAMP5 median fres of Table S3 (1.09 Hz): about half of the LAMP5 cells here have
their peak at the lowest frequency.

## Models and tests used below

Mixed models with a random intercept for donor (human) or mouse, fitted by REML. Reading (a)
of O5 gives Satterthwaite t and F tests (`R.lmm_satterthwaite`, the method of SPSS MIXED and
lmerTest); reading (b) gives statsmodels MixedLM Wald tests (z, or chi-square for several
coefficients). Terms are tested as Type III terms: subclass is sum-coded, so a main effect is
the effect averaged over subclasses. Vm (O4) enters where the criterion names it: reading (a)
the pre-chirp potential for chirp measures, reading (b) the pre-step potential (the only one
for cells without a chirp, so it is the main reading for sag and input resistance). Cells
whose input resistance under the reading used is not positive (a fitting failure on noisy
steps) are left out of tests that use it; their count is printed.

In [ ]:
def mixed(formula, data, group, tests):
    """Reading a (Satterthwaite) and reading b (statsmodels Wald) of the same model."""
    d = data.dropna().copy()     # callers pass only the columns the model uses
    sat = R.lmm_satterthwaite(formula, d, group, tests)
    try:
        f = smf.mixedlm(formula, d, groups=d[group]).fit(reml=True)
        conv = bool(f.converged)
    except Exception as e:     # recorded as a failed fit, never as a result
        f, conv = None, f"failed: {e}"
    out = dict(n=sat["n"], groups=sat["groups"], conv_a=sat["converged"], conv_b=conv)
    for k, keys in tests.items():
        t = sat["tests"][k]
        out[f"{k} a"] = (round(t["est"], 5) if len(keys) == 1 else f"F={t['F']:.2f}", f"df={t['df1']},{t['df2']:.1f}", float(f"{t['p']:.3g}"))
        if f is not None:
            idx = [list(f.params.index).index(c) for c in keys]
            bvec = f.params.values[idx]
            V = f.cov_params().values[np.ix_(idx, idx)]
            w = float(bvec @ np.linalg.solve(V, bvec))
            out[f"{k} b"] = float(f"{stats.chi2.sf(w, len(keys)):.3g}")
    return out

def mwu(x, y):
    x, y = pd.Series(x).dropna(), pd.Series(y).dropna()
    if len(x) == 0 or len(y) == 0:
        return dict(nx=len(x), ny=len(y), U=np.nan, p=np.nan, med_x=np.nan, med_y=np.nan)
    r = stats.mannwhitneyu(x, y, alternative="two-sided")
    return dict(nx=len(x), ny=len(y), U=r.statistic, p=float(f"{r.pvalue:.3g}"), med_x=round(x.median(), 4), med_y=round(y.median(), 4))

def cols_of(data, formula):
    import patsy
    return list(patsy.dmatrices(formula, data.dropna(subset=["y"]), return_type="dataframe")[1].columns)

pos_rin = lambda d, sr: d[d[f"rin_{sr}"] > 0]
for sr in "ab":
    print(f"cells with a sag ratio but non-positive input resistance, reading {sr}:",
          allc[allc["sag_ok"] & (allc[f"rin_{sr}"] <= 0)].groupby(["species", "prep"]).size().to_dict())
SUBS = ["LAMP5", "VIP", "PV", "SST"]

### Human against mouse: shared code

The public mouse cells are LAMP5 and VIP cells only (the Table S2b PV and SST cells are not in
000020; see mouse-specimen-link), so every species comparison of PV or SST cells is missing,
and the species by subclass models are fitted on LAMP5 and VIP cells (one-df interaction).
Mouse subclass is the T-type's subclass, with Sncg types counted as VIP (reading a: the
coverage of the paper's counts is then even, LAMP5 374 of 435 and VIP 86 of 101) or as LAMP5
(reading b, shown).

In [ ]:
def species_frame(meas, rd, vm, chirp=True, sncg="a"):
    d = allc[(allc["prep"] == "acute") & allc["sub"].notna()].copy()
    if sncg == "b":
        d.loc[d["species"] == "mouse", "sub"] = d.loc[d["species"] == "mouse", "sub_sncg_lamp5"]
    d = d[d["chirp_ok"]] if chirp else d[d["sag_ok"]]
    if meas == "rin":
        d = d[d[f"rin_{rd}"] > 0]
    d = d.assign(y=d[f"{meas}_{rd}"], vm=d[f"vm_{vm}"], g=d["species"] + ":" + d["donor"].astype(str))
    return d

def species_tests(meas, rd, vm, chirp=True, sncg="a"):
    d = species_frame(meas, rd, vm, chirp, sncg)
    rows = {}
    for sb in SUBS:
        rows[sb] = mwu(d.loc[(d.species == "human") & (d["sub"] == sb), "y"], d.loc[(d.species == "mouse") & (d["sub"] == sb), "y"])
    lv = d[d["sub"].isin(["LAMP5", "VIP"])][["y", "species", "sub", "vm", "g"]]
    form = "y ~ C(species, Sum) * C(sub, Sum) + vm"
    cs = cols_of(lv, form)
    inter = [c for c in cs if ":" in c]
    sp = [c for c in cs if c.startswith("C(species") and ":" not in c]
    model = mixed(form, lv, "g", {"interaction": inter, "species": sp})
    return pd.DataFrame(rows).T, model, d

def show_species(meas, readings, chirp=True):
    res = {}
    for rd, vm, sncg in readings:
        mw_, model, d = species_tests(meas, rd, vm, chirp, sncg)
        res[(rd, vm, sncg)] = (mw_, model)
        print(f"--- {meas}, reading {rd}, Vm {vm}, Sncg {sncg}: Mann-Whitney human (x) against mouse (y)")
        print(mw_.to_string())
        print("    mixed model on LAMP5 and VIP:", model)
    return res

## fig2b-human-higher-fres-q

> Human cortical interneurons have higher peak resonance frequencies and resonance strengths
> than mouse interneurons (Kolmogorov-Smirnov tests, p < 0.001 each).

Two-sample Kolmogorov-Smirnov tests, human acute against mouse cells with a chirp. Reproduced
when both give p < 0.001 and the human median exceeds the mouse median for both. The mouse
sample has no PV or SST cells; the test restricted to LAMP5 and VIP cells in both species is
shown as a variant.

In [ ]:
ha, mo = acute[acute.chirp_ok], mouse[mouse.chirp_ok]
for label, h_, m_ in [("all subclasses", ha, mo), ("LAMP5 and VIP only", ha[ha["sub"].isin(["LAMP5", "VIP"])], mo)]:
    for rd in "ba":
        for v in ("fres", "Q"):
            k = stats.ks_2samp(h_[f"{v}_{rd}"], m_[f"{v}_{rd}"])
            print(f"{label}, reading {rd}, {v}: D = {k.statistic:.3f}, p = {k.pvalue:.3g}; medians human {h_[f'{v}_{rd}'].median():.3f}"
                  f" (n {len(h_)}), mouse {m_[f'{v}_{rd}'].median():.3f} (n {len(m_)}); means {h_[f'{v}_{rd}'].mean():.3f} and {m_[f'{v}_{rd}'].mean():.3f}")
print("paper: fres D = 0.276, p = 1.3e-09; Q D = 0.274, p = 2.1e-09 (297 human, 248 mouse)")

**Result: reproduced, under both chirp readings.** fres and Q differ between human acute (212) and mouse (139) cells: fres D = 0.239, p = 1.0e-4; Q D = 0.244, p = 6.6e-5, with human medians above mouse (fres 1.20 against 0.40 Hz, Q 1.010 against 1.000). Reading (a) gives the same (p = 4.7e-7 and 1.5e-5), so the outcome does not rest on the chirp reading. The mouse sample has no PV or SST cells; with LAMP5 and VIP cells only in both species, fres still differs (p = 2.8e-4) but Q does not (p = 0.25), with equal medians.

## fig2ab-peak-range

> Both human and mouse interneurons show resonance peaks roughly in the 2 to 5 Hz range.

fres of resonant cells (fres > 1 Hz, Q > 1.05) per species. Reading (a) of the criterion: the
median lies within 1.8 to 5.5 Hz; reading (b): the interquartile range does.

In [ ]:
for rd in "ba":
    for nm, d in [("human acute", ha), ("mouse", mo)]:
        r = d[R.is_resonant(d[f"fres_{rd}"], d[f"Q_{rd}"])][f"fres_{rd}"]
        q1, q2, q3 = r.quantile([0.25, 0.5, 0.75])
        print(f"chirp reading {rd}, {nm}: {len(r)} resonant cells, median {q2:.2f} Hz, IQR {q1:.2f} to {q3:.2f} Hz, "
              f"range {r.min():.2f} to {r.max():.2f}; median within 1.8-5.5: {1.8 <= q2 <= 5.5}; IQR within: {q1 >= 1.8 and q3 <= 5.5}")

**Result: not reproduced.** Among resonant cells, the human median fres is 2.58 Hz (IQR 1.93 to 3.09, 54 cells) and the mouse median 1.70 Hz (IQR 1.45 to 2.63, 11 cells) under chirp reading (b). The human cells meet both readings of the criterion; the mouse median falls below 1.8 Hz, and under chirp reading (a) too (1.75 Hz, 14 cells). Only 11 mouse cells are resonant, and the mouse PV cells are missing.

## fig2c-resonant-proportion

> A larger proportion of human than mouse interneurons are resonant (fres > 1 Hz and Q > 1.05),
> 32.7% against 9.3%.

Fractions over cells with a chirp; chi-square under both readings of O6 (a: goodness of fit of
the human counts against the mouse fraction; b: 2x2 Pearson without correction). Reproduced
when both fractions are within tolerance (32.7 +/- 3.27%, 9.3 +/- 0.93%) and the test gives
p < 0.001.

In [ ]:
for rd in "ba":
    kh, nh = int(R.is_resonant(ha[f"fres_{rd}"], ha[f"Q_{rd}"]).sum()), len(ha)
    km, nm_ = int(R.is_resonant(mo[f"fres_{rd}"], mo[f"Q_{rd}"]).sum()), len(mo)
    print(f"chirp reading {rd}: human {kh}/{nh} = {100 * kh / nh:.1f}%, mouse {km}/{nm_} = {100 * km / nm_:.1f}%;",
          "O6 a chi2 = %.2f, p = %.3g;" % R.chi2_gof(kh, nh, km / nm_), "O6 b chi2 = %.2f, p = %.3g" % R.chi2_2x2(kh, nh, km, nm_))
    hl = ha[ha["sub"].isin(["LAMP5", "VIP"])]
    kl = int(R.is_resonant(hl[f"fres_{rd}"], hl[f"Q_{rd}"]).sum())
    print(f"   variant, human LAMP5 and VIP only: {kl}/{len(hl)} = {100 * kl / len(hl):.1f}%")

**Result: not reproduced, under the assumed chirp reading (b).** 25.5% of human acute cells (54 of 212) and 7.9% of mouse cells (11 of 139) are resonant; both chi-square tests give p < 0.001 (goodness of fit p = 2.8e-21, 2x2 p = 3.5e-5), but neither fraction is within tolerance (32.7 +/- 3.27%, 9.3 +/- 0.93%). Under chirp reading (a) the fractions are 31.1% and 10.1%, both within tolerance, with p = 2.2e-24 and 4.2e-6, which would reproduce the claim.

## fig2d-subclass-proportion

> Human LAMP5, VIP and PV interneurons are more often resonant than their mouse counterparts
> (p < 0.001 each).

Per subclass, the goodness-of-fit chi-square of the human counts against the mouse fraction of
that subclass (O6 a). Reproduced when the human fractions are within tolerance (LAMP5 31.25
+/- 3.125%, VIP 25.0 +/- 2.5%, PV 43.6 +/- 4.36%) and each test gives p < 0.001 with the human
fraction higher. The mouse PV cells are not public, so the PV test cannot be run.

In [ ]:
for rd in "ba":
    for sncg in "ab":
        mo_ = mo.assign(sub=mo["sub"] if sncg == "a" else mo["sub_sncg_lamp5"])
        out = {}
        for sb in SUBS:
            h_, m_ = ha[ha["sub"] == sb], mo_[mo_["sub"] == sb]
            kh = int(R.is_resonant(h_[f"fres_{rd}"], h_[f"Q_{rd}"]).sum())
            km = int(R.is_resonant(m_[f"fres_{rd}"], m_[f"Q_{rd}"]).sum())
            pm = km / len(m_) if len(m_) else np.nan
            chi = R.chi2_gof(kh, len(h_), pm) if len(m_) else (np.nan, np.nan)
            out[sb] = f"human {kh}/{len(h_)} = {100 * kh / len(h_):.1f}%, mouse {km}/{len(m_)}" + \
                      (f" = {100 * pm:.1f}%, chi2 = {chi[0]:.2f}, p = {chi[1]:.3g}" if len(m_) else " (no mouse cells)")
        print(f"chirp reading {rd}, Sncg {sncg}:")
        for k, v in out.items():
            print("   ", k, v)
print("Table S3: LAMP5 45/144 vs mouse expected, chi2 84.34; VIP 15/60, chi2 158.20; PV 34/78, chi2 57.53")

**Result: not reproduced.** The criterion has no partial rule and asks for all three fractions within tolerance and each test at p < 0.001. Under chirp reading (b), human LAMP5 (19.5%, 17 of 87) and VIP cells (12.2%, 6 of 49) are more often resonant than mouse LAMP5 (10.0%) and VIP cells (2.6%), at p = 0.003 and 1.8e-5; the LAMP5 test misses p < 0.001 and the LAMP5 and VIP fractions are below tolerance (31.25 +/- 3.125%, 25.0 +/- 2.5%). The human PV fraction (43.1%) is within tolerance, but mouse PV cells are not public, so the PV test cannot be run. Under chirp reading (a) LAMP5 29.9% and PV 46.2% are within tolerance, VIP 16.3% is not, and both tests give p < 1e-5. Under the main reading no subclass meets both its tolerance and its test (LAMP5 neither, VIP the test only, PV untestable), so the criterion is not met. It is not met under chirp reading (a) or at LOWESS fractions 0.02 and 0.08 either.

## fig2d-pv-highest

> Among human interneuron subclasses, PV cells have the highest proportion of resonant cells.

Reproduced when the human PV resonant fraction exceeds those of LAMP5, VIP and SST.

In [ ]:
for rd in "ba":
    fr = ha.assign(r=R.is_resonant(ha[f"fres_{rd}"], ha[f"Q_{rd}"])).groupby("sub")["r"].agg(["sum", "count", "mean"])
    print(f"chirp reading {rd}:", {k: f"{int(v['sum'])}/{int(v['count'])} = {100 * v['mean']:.1f}%" for k, v in fr.iterrows()},
          "| PV highest:", fr["mean"].idxmax() == "PV")
    for src in ("000630", "000636"):
        f2 = ha[ha.source == src].assign(r=R.is_resonant(ha[f"fres_{rd}"], ha[f"Q_{rd}"])).groupby("sub")["r"].mean().round(3).to_dict()
        print(f"   by source {src}:", f2)

**Result: reproduced.** PV cells have the highest resonant fraction among human acute subclasses: 43.1% against LAMP5 19.5%, SST 27.3% and VIP 12.2% under chirp reading (b); 46.2% against 29.9%, 18.2% and 16.3% under reading (a).

## figs2a-stringent-criteria

> Human interneurons are more often resonant than mouse interneurons also under the stricter
> criteria fres > 1 Hz and Q > 1.1 or Q > 1.2.

In [ ]:
for rd in "ba":
    for q in (1.05, 1.1, 1.2):
        fh = R.is_resonant(ha[f"fres_{rd}"], ha[f"Q_{rd}"], q).mean()
        fm = R.is_resonant(mo[f"fres_{rd}"], mo[f"Q_{rd}"], q).mean()
        hl = ha[ha["sub"].isin(["LAMP5", "VIP"])]
        fl = R.is_resonant(hl[f"fres_{rd}"], hl[f"Q_{rd}"], q).mean()
        print(f"chirp reading {rd}, Q > {q}: human {100 * fh:.1f}%, mouse {100 * fm:.1f}% (human LAMP5 and VIP only {100 * fl:.1f}%)")

**Result: reproduced.** Human cells are more often resonant than mouse cells at Q > 1.1 (7.1% against 2.2%) and Q > 1.2 (1.4% against 0.7%) under chirp reading (b), and under reading (a) (12.7% against 2.9%; 1.9% against 0.0%). At Q > 1.2 these are 3 human and 1 mouse cell.

## fig2e-fres-species-subclass

> Species differences in peak resonance frequency depend on subclass (species by subclass
> interaction, p = 0.006); human PV and VIP interneurons have higher fres than mouse.

Mixed model of fres on species, subclass, their interaction and Vm with donor intercept (on
LAMP5 and VIP cells, the subclasses both species have), and two-sided Mann-Whitney U per
subclass. Reproduced when the interaction has p < 0.05, the PV and VIP tests give p < 0.05 with
human higher, and the medians are within tolerance (human PV 2.18 +/- 0.218, mouse PV 1.30,
human VIP 0.95 +/- 0.095, mouse VIP 0.40 +/- 0.04 Hz). The PV comparison cannot be run.

In [ ]:
fig2e = show_species("fres", [("b", "a", "a"), ("b", "b", "a"), ("a", "a", "a"), ("b", "a", "b")])

**Result: partially reproduced, under the assumed chirp reading (b).** On LAMP5 and VIP cells the species by subclass interaction is significant (Satterthwaite p = 0.004, Wald p = 0.004; Vm reading (b) p = 0.003, chirp reading (a) p = 0.006). Human VIP cells have higher fres than mouse VIP cells (p = 0.0005), with both medians at the floor (0.40 Hz). The PV comparison cannot be run. Of the medians, mouse VIP (0.40 Hz) is within tolerance; human PV (2.50 Hz against 2.18 +/- 0.218) and human VIP (0.40 against 0.95 +/- 0.095) are not. With Sncg counted as LAMP5 the interaction gives p = 0.086.

## fig2e-lamp5-sst-fres-similar

> Human and mouse LAMP5 and SST interneurons do not differ in peak resonance frequency.

A claim of absence, judged on two-sided Mann-Whitney U per subclass at p < 0.05 (rows LAMP5
and SST of the tables above). The mouse SST cells are not public.

In [ ]:
for k, (mw_, model) in fig2e.items():
    print(f"reading {k}: LAMP5", mw_.loc["LAMP5"].to_dict(), "| SST", mw_.loc["SST"].to_dict())
print("Table S3: LAMP5 U = 7717, p = 0.133 (medians 1.09 and 1.15 Hz); SST p = 0.701 (0.40 and 0.40 Hz)")

**Result: partially reproduced, under the assumed chirp reading (b).** Human and mouse LAMP5 cells do not differ in fres (p = 0.24; medians 0.40 and 1.15 Hz). The mouse SST cells are not public, so the SST part cannot be tested. Under chirp reading (a) the LAMP5 cells differ (p = 0.009).

## fig2f-f3db-species-subclass

> Species differences in 3 dB cutoff frequency are subclass-specific (p < 0.001); human LAMP5,
> PV and SST interneurons have lower 3 dB cutoffs than mouse, while VIP cells are similar.

Model and tests as for fig2e with f3dB. Reproduced when the interaction has p < 0.05, LAMP5, PV
and SST each show human lower at p < 0.05, VIP does not differ, and the medians are within
tolerance (human LAMP5 7.36 +/- 0.736, mouse LAMP5 15.20 +/- 1.52, human PV 13.03 +/- 1.303,
mouse PV 20.86 Hz); partially reproduced when the interaction and two of the three lower
subclasses hold. PV and SST cannot be compared.

In [ ]:
fig2f = show_species("f3db", [("b", "a", "a"), ("b", "b", "a"), ("a", "a", "a"), ("b", "a", "b")])

**Result: partially reproduced.** On LAMP5 and VIP cells the species by subclass interaction is significant (p = 7e-14), human LAMP5 cells have a lower f3dB than mouse (6.71 against 15.18 Hz, p = 5e-27), and VIP cells do not differ (p = 0.067). The medians that can be measured are within tolerance (human LAMP5 6.71, mouse LAMP5 15.18, human PV 13.66 Hz). The PV and SST comparisons cannot be run, as the mouse cells are not public. The same holds under every reading shown.

## fig2g-q-species-subclass

> Human PV, VIP and LAMP5 interneurons, but not SST, show higher resonance strength than mouse
> counterparts.

Reproduced when PV, VIP and LAMP5 each show human higher at p < 0.05 by two-sided Mann-Whitney
U, SST does not differ, and the species effect of the mixed model has p < 0.05; partially
reproduced when two of the three subclasses hold. Medians against Table S3 (human PV 1.04 +/-
0.004, mouse PV 1.008, human LAMP5 1.013 +/- 0.0013).

In [ ]:
fig2g = show_species("Q", [("b", "a", "a"), ("b", "b", "a"), ("a", "a", "a"), ("b", "a", "b")])

**Result: not reproduced.** Under chirp reading (b), human VIP cells have a higher Q than mouse VIP cells (p = 0.001), but LAMP5 cells do not differ (p = 0.77), and the species effect of the mixed model on LAMP5 and VIP cells is not significant (p = 0.56). PV and SST cannot be compared. The human PV median Q (1.041) is within tolerance; the human LAMP5 median (1.000) is not (1.013 +/- 0.0013). Chirp reading (a) gives the same pattern (LAMP5 p = 0.25, species p = 0.82).

## fig3c-sag-species

> Species differences in sag ratio are subclass-specific (p < 0.001), with higher sag ratios in
> human LAMP5, VIP, PV and SST interneurons than in mouse.

All cells with a sag ratio (not only chirp cells, as the paper's counts show). Vm reading (b),
the pre-step potential, as reading (a) exists only for chirp cells. Reproduced when the
interaction has p < 0.05, all four subclasses show human higher at p < 0.05, and the medians are
within tolerance (human LAMP5 0.113, mouse LAMP5 0.044, human VIP 0.177, human PV 0.106, each
+/- 10%) under one reading of O2; partially reproduced when three subclasses hold.

In [ ]:
fig3c = show_species("sag", [("a", "b", "a"), ("b", "b", "a"), ("a", "b", "b")], chirp=False)
for rd in "ab":
    print(f"human medians, sag reading {rd}:", acute[acute.sag_ok].groupby("sub")[f"sag_{rd}"].median().round(4).to_dict())

**Result: partially reproduced.** Human LAMP5 (0.112 against 0.044) and VIP cells (0.175 against 0.051) have higher sag ratios than mouse cells (p = 8e-35 and 2e-16), the species by subclass interaction on these two subclasses is significant (p = 0.007), and the four medians are within tolerance (human LAMP5 0.112, mouse LAMP5 0.044, human VIP 0.175, human PV 0.106). PV and SST cannot be compared, as the mouse cells are not public. The same holds under sag reading (b) and with Sncg counted as LAMP5.

## fig3d-input-resistance-species

> Species differences in input resistance are subclass-specific (p < 0.001); human LAMP5 and PV
> interneurons have higher input resistance than mouse, VIP and SST do not differ.

As fig3c with input resistance. Reproduced when the interaction has p < 0.05, LAMP5 and PV show
human higher at p < 0.05, VIP and SST give p >= 0.05, and the medians are within tolerance
(human LAMP5 231.26 +/- 23.13, mouse LAMP5 125.87 +/- 12.59, human PV 151.45 +/- 15.15, mouse PV
112.35 MOhm) under one reading of O3.

In [ ]:
fig3d = show_species("rin", [("a", "b", "a"), ("b", "b", "a"), ("a", "b", "b")], chirp=False)
for rd in "ab":
    print(f"human medians, input resistance reading {rd}:", acute[acute.sag_ok].groupby("sub")[f"rin_{rd}"].median().round(2).to_dict())

**Result: partially reproduced.** The species by subclass interaction on LAMP5 and VIP cells is significant (p = 1e-19), human LAMP5 cells have a higher input resistance than mouse (258 against 126 MOhm, p = 5e-69), and VIP cells do not differ (p = 0.13). PV and SST cannot be compared. The mouse LAMP5 (126.4) and human PV medians (142.0 MOhm) are within tolerance under reading (a); the human LAMP5 median (258.4) is above 231.26 +/- 23.13 under both readings of O3.

### Sag ratio and input resistance against resonance: shared code

Model of the resonance measure on sag, input resistance, subclass, sag by subclass and input
resistance by subclass, with donor intercept (Fig. 3 legend; no Vm term), then per subclass
the measure on sag and input resistance. Every combination of chirp reading (O1) and sag and
input resistance reading (O2, O3) is fitted, each under O5 readings (a) Satterthwaite and (b)
Wald z. Convergence of every fit is printed (conv_a, conv_b). GEE with an exchangeable
correlation within donor is shown for the full model under the main readings.

In [ ]:
def fig3_frame(d, meas, rd, sr):
    d = d[d["chirp_ok"] & d["sub"].notna() & (d[f"rin_{sr}"] > 0)]
    return pd.DataFrame(dict(y=d[f"{meas}_{rd}"], sag=d[f"sag_{sr}"], rin=d[f"rin_{sr}"], sub=d["sub"], g=d["donor"].astype(str))).dropna()

def fig3_fits(d0, meas, reduced=False):
    rows = []
    for rd in "ba":
        for sr in "ab":
            d = fig3_frame(d0, meas, rd, sr)
            form = "y ~ sag * C(sub, Sum) + rin * C(sub, Sum)"
            cs = cols_of(d, form)
            tests = {"sag": ["sag"], "rin": ["rin"], "sag x sub": [c for c in cs if c.startswith("sag:")],
                     "rin x sub": [c for c in cs if c.startswith("rin:")]}
            row = dict(chirp=rd, sag_rin=sr, **{f"full {k}": v for k, v in mixed(form, d, "g", tests).items()})
            if reduced:
                row.update({f"reduced {k}": v for k, v in mixed("y ~ sag + rin + C(sub, Sum)", d, "g", {"sag": ["sag"], "rin": ["rin"]}).items()
                            if k not in ("n", "groups")})
            for sb, gg in d.groupby("sub"):
                r = mixed("y ~ sag + rin", gg, "g", {"sag": ["sag"], "rin": ["rin"]})
                row.update({f"{sb} {k}": v for k, v in r.items()})
            rows.append(row)
    return pd.DataFrame(rows)

def gee_full(d0, meas, rd="b", sr="a"):
    d = fig3_frame(d0, meas, rd, sr)
    try:
        f = smf.gee("y ~ sag * C(sub, Sum) + rin * C(sub, Sum)", groups=d["g"], data=d, cov_struct=sm.cov_struct.Exchangeable()).fit()
        return {k: (round(float(f.params[k]), 5), float(f"{f.pvalues[k]:.3g}")) for k in ["sag", "rin"]}
    except Exception as e:
        return f"GEE failed: {e}"

def show_fit(t, keys):
    for _, r in t.iterrows():
        print(f"chirp {r['chirp']}, sag/rin {r['sag_rin']}:")
        for k in keys:
            print(f"   {k}:", {c.replace(k + ' ', ''): r[c] for c in t.columns if c.startswith(k + " ")})

## fig3eg-human-fres-sag-rin

> In human interneurons, sag ratio and input resistance predict peak resonance frequency in a
> subclass-specific way; fres rises with sag ratio in LAMP5 (beta 3.31), VIP (4.79) and PV
> (6.13) cells and falls with input resistance in LAMP5 (-0.002) and PV (-0.006) cells.

Human acute cells of the four subclasses (000630 and 000636). Reproduced when both
interactions have p < 0.05, the five per-subclass slopes have the stated sign with p < 0.05,
and the slopes are within tolerance; partially reproduced when the signs and p-values hold but
a slope does not, or when one interaction misses. Main readings: chirp (b), sag and input
resistance (a), O5 (a).

In [ ]:
fig3eg = fig3_fits(acute, "fres")
show_fit(fig3eg, ["full", "LAMP5", "VIP", "PV", "SST"])
print("GEE, full model, main readings:", gee_full(acute, "fres"))
print("cells per subclass (main readings):", fig3_frame(acute, "fres", "b", "a")["sub"].value_counts().to_dict(),
      "donors:", fig3_frame(acute, "fres", "b", "a").groupby("sub")["g"].nunique().to_dict())
print("expected: LAMP5 sag 3.31, VIP 4.79, PV 6.13 (+/- 10%); LAMP5 rin -0.002, PV -0.006; interactions p = 0.035 and 0.029")

**Result: not reproduced.** On 212 human acute cells of four subclasses (main readings), fres rises with sag in LAMP5 (b = 3.62, p = 8.5e-5), VIP (4.34, p = 3.3e-5) and PV cells (4.44, p = 0.006), and falls with input resistance in PV cells (b = -0.0048, p = 0.023); LAMP5 input resistance has the stated sign but p = 0.094. Neither interaction with subclass is significant (sag p = 0.90, input resistance p = 0.57; paper 0.035 and 0.029). The LAMP5 and VIP sag slopes and the LAMP5 input resistance slope are within tolerance; the PV slopes are not (4.44 against 6.13, -0.0048 against -0.006). In every combination of readings both interactions miss and LAMP5 input resistance gives p >= 0.05 (0.052 to 0.094), so the criterion's partial rule (one interaction missing) is not met.

## fig3ik-human-q-sag-rin

> In human interneurons, sag ratio strongly predicts resonance strength (beta 0.47, p < 0.001)
> with no subclass difference, and input resistance is negatively related to it (beta -0.0001,
> p = 0.005).

The slopes are those of a model without the interactions (criterion), the full model's shown.
Reproduced when sag has a positive effect at p < 0.001, its interaction with subclass gives p
>= 0.05, input resistance a negative effect at p < 0.05, and the slopes are within tolerance
(0.47 +/- 0.047, -0.0001 +/- 0.00001); partially reproduced when the tests hold but a slope
does not.

In [ ]:
fig3ik = fig3_fits(acute, "Q", reduced=True)
show_fit(fig3ik, ["reduced", "full"])
print("GEE, full model, main readings:", gee_full(acute, "Q"))

**Result: partially reproduced, under the assumed chirp reading (b).** In the model without interactions Q rises with sag (b = 0.31, Satterthwaite p = 0.0004) and falls with input resistance (b = -0.00024, p = 0.002), and the sag by subclass interaction is not significant (p = 0.099), the three tests the criterion names; both slopes are outside tolerance (0.47 +/- 0.047, -0.0001 +/- 0.00001). The result rests on the readings: with sag and input resistance reading (b) sag gives p = 0.022, and under chirp reading (a) neither sag (p = 0.13) nor input resistance (p = 0.26) is significant. GEE on the full model gives sag p = 0.17.

## fig3fh-mouse-fres

> In mouse interneurons, sag ratio does not predict peak resonance frequency (p = .113); only
> input resistance does, with no difference between subclasses.

Model as for fig3eg on the mouse cells (LAMP5 and VIP). Reproduced when the sag main effect
gives p >= 0.05, the input resistance main effect p < 0.05, and its interaction with subclass
p >= 0.05.

In [ ]:
fig3fh = fig3_fits(mouse, "fres")
show_fit(fig3fh, ["full"])

**Result: not reproduced.** In mouse cells (139, LAMP5 and VIP) sag does not predict fres at p < 0.05 (p = 0.056) and the input resistance by subclass interaction is not significant (p = 0.41), but input resistance does not predict fres either (p = 0.93), where the paper finds p = 0.023. Under the other readings sag reaches p < 0.05 (0.006 to 0.046) and input resistance stays at p >= 0.17.

## fig3jl-mouse-q-sag-rin

> In mouse interneurons, sag ratio is only weakly related to resonance strength (beta 0.018,
> p = 0.021) and input resistance is not related to it (p = 0.820).

Model as for fig3ik on the mouse cells. Reproduced when sag has a positive effect at p < 0.05
and input resistance gives p >= 0.05; the sag slope is reported against 0.018.

In [ ]:
fig3jl = fig3_fits(mouse, "Q", reduced=True)
show_fit(fig3jl, ["reduced", "full"])
q = mouse[mouse.chirp_ok]["Q_b"]
print("mouse Q, reading b: largest values", q.nlargest(3).round(3).tolist(), "| without the largest:")
fig3jl_out = fig3_fits(mouse[~mouse["Q_b"].eq(q.max())], "Q", reduced=True).query("chirp == 'b' and sag_rin == 'a'")
show_fit(fig3jl_out, ["reduced"])

**Result: reproduced.** In mouse cells Q rises with sag (b = 0.21, p = 0.004) and is not related to input resistance (p = 0.56), under every combination of readings (sag p = 0.004 to 0.016, input resistance p >= 0.39). The sag slope is about ten times the paper's 0.018. One mouse cell has Q = 16.8; without it the sag slope is 0.11 (p = 0.011).

## fig5g-culture-sag

> Cultured LAMP5 and PV interneurons have higher sag ratios than acutely recorded ones, while
> input resistance does not differ between preparations.

Human cultured cells of 000636 against the human acute cells. Sag by two-sided Mann-Whitney U
per subclass; input resistance in the mixed model of preparation, subclass, their interaction
and Vm (pre-step potential, reading b of O4, as most cells have no chirp) with donor
intercept. Reproduced when LAMP5 and PV sag are higher in culture at p < 0.001, the preparation
effect on input resistance gives p >= 0.05, and the cultured medians are within tolerance
(LAMP5 0.26 +/- 0.026, PV 0.28 +/- 0.028).

In [ ]:
hs = hum[hum["sag_ok"] & hum["sub"].notna()]
for sr in "ab":
    print(f"sag reading {sr}, cultured (x) against acute (y):")
    print(pd.DataFrame({sb: mwu(hs.loc[(hs.prep == "culture") & (hs["sub"] == sb), f"sag_{sr}"],
                                hs.loc[(hs.prep == "acute") & (hs["sub"] == sb), f"sag_{sr}"]) for sb in SUBS}).T.to_string())
for sr in "ab":
    d = hs[hs[f"rin_{sr}"] > 0]
    d = pd.DataFrame(dict(y=d[f"rin_{sr}"], prep=d["prep"], sub=d["sub"], vm=d["vm_b"], g=d["donor"].astype(str))).dropna()
    form = "y ~ C(prep, Sum) * C(sub, Sum) + vm"
    cs = cols_of(d, form)
    r = mixed(form, d, "g", {"prep": [c for c in cs if c.startswith("C(prep") and ":" not in c],
                             "prep x sub": [c for c in cs if ":" in c]})
    print(f"input resistance reading {sr}, mixed model:", r)
    print("   per subclass, cultured against acute:", {sb: mwu(d.loc[(d.prep == "culture") & (d["sub"] == sb), "y"],
                                                         d.loc[(d.prep == "acute") & (d["sub"] == sb), "y"])["p"] for sb in SUBS})
print("donors with both acute and cultured cells:", len(set(acute["donor"]) & set(cult["donor"])))

**Result: partially reproduced.** Cultured LAMP5 (0.256 against 0.112, p = 3.8e-6) and PV cells (0.283 against 0.106, p = 1e-35) have higher sag ratios than acute ones, and the cultured medians are within tolerance (0.26 +/- 0.026, 0.28 +/- 0.028). But input resistance differs between preparations in the mixed model (p = 0.004 under reading (a), 0.0002 under reading (b)), where the paper finds p = 0.095.

## fig5i-culture-proportion

> A higher proportion of interneurons is resonant in organotypic slice culture than in acute
> slices (60.8% against 32.7%), in LAMP5, PV and SST interneurons but not VIP.

Overall test under both readings of O6; per subclass the goodness-of-fit chi-square of the
cultured counts against the acute fraction (O6 a). Reproduced when both fractions are within
tolerance (60.8 +/- 6.08%, 32.7 +/- 3.27%), the overall test gives p < 0.001, LAMP5, PV and SST
are higher in culture at p < 0.05, and VIP gives p >= 0.05.

In [ ]:
hc = hum[hum["chirp_ok"] & hum["sub"].notna()]
for rd in "ba":
    r_ = R.is_resonant(hc[f"fres_{rd}"], hc[f"Q_{rd}"])
    kc, nc = int(r_[hc.prep == "culture"].sum()), int((hc.prep == "culture").sum())
    ka, na = int(r_[hc.prep == "acute"].sum()), int((hc.prep == "acute").sum())
    print(f"chirp reading {rd}: cultured {kc}/{nc} = {100 * kc / nc:.1f}%, acute {ka}/{na} = {100 * ka / na:.1f}%;",
          "O6 a chi2 = %.2f, p = %.3g;" % R.chi2_gof(kc, nc, ka / na), "O6 b chi2 = %.2f, p = %.3g" % R.chi2_2x2(kc, nc, ka, na))
    for sb in SUBS:
        c_, a_ = hc[(hc.prep == "culture") & (hc["sub"] == sb)], hc[(hc.prep == "acute") & (hc["sub"] == sb)]
        k1 = int(R.is_resonant(c_[f"fres_{rd}"], c_[f"Q_{rd}"]).sum())
        k2 = int(R.is_resonant(a_[f"fres_{rd}"], a_[f"Q_{rd}"]).sum())
        chi = R.chi2_gof(k1, len(c_), k2 / len(a_))
        print(f"   {sb}: cultured {k1}/{len(c_)} = {100 * k1 / len(c_):.1f}%, acute {k2}/{len(a_)} = {100 * k2 / len(a_):.1f}%, "
              f"chi2 = {chi[0]:.2f}, p = {chi[1]:.3g}")
print("paper: cultured LAMP5 13/21, PV 104/125, SST 39/94, VIP 7/28 (163/268)")

**Result: partially reproduced, under the assumed chirp reading (b).** 63.0% of cultured cells (114 of 181) and 25.5% of acute cells are resonant (p = 5e-31 and 7e-14); the cultured fraction is within tolerance and the acute one is not (32.7 +/- 3.27%). LAMP5 (p = 8e-5) and PV (p = 2e-17) are higher in culture and VIP is not (p = 0.78), but SST misses (p = 0.055). Under chirp reading (a) every part holds (65.2% and 31.1%; SST p = 6e-7; VIP p = 0.35), which would reproduce the claim.

## fig5jk-culture-fres-q

> Peak resonance frequency (LAMP5, PV) and resonance strength (LAMP5, PV, SST) are higher in
> cultured than in acute interneurons.

Two-sided Mann-Whitney U per subclass, cultured against acute. Reproduced when the five
comparisons show culture higher at p < 0.05; partially reproduced when four do. Cultured medians
against Table S3 (LAMP5 fres 2.40 +/- 0.24 Hz, PV Q 1.17 +/- 0.017).

In [ ]:
for rd in "ba":
    print(f"chirp reading {rd}, cultured (x) against acute (y):")
    for v in ("fres", "Q", "f3db"):
        t = pd.DataFrame({sb: mwu(hc.loc[(hc.prep == "culture") & (hc["sub"] == sb), f"{v}_{rd}"],
                                  hc.loc[(hc.prep == "acute") & (hc["sub"] == sb), f"{v}_{rd}"]) for sb in SUBS}).T
        print(f"  {v}:"); print(t.to_string())

**Result: partially reproduced.** Four of the five comparisons show culture higher: fres in LAMP5 (p = 0.004) and PV (p = 3e-7), Q in LAMP5 (p = 0.002) and PV (p = 2e-13); Q in SST does not differ (p = 0.33). The cultured medians are outside tolerance (LAMP5 fres 2.75 Hz against 2.40 +/- 0.24, from 8 cells; PV Q 1.188 against 1.17 +/- 0.017). Chirp reading (a) gives the same four (SST Q p = 0.11).

## fig5l-culture-f3db-same

> The 3 dB cutoff frequency does not differ between acute and cultured interneurons.

A claim of absence: two-sided Mann-Whitney U per subclass at p < 0.05 (the f3db rows of the
tables above). Reproduced when no subclass reaches p < 0.05.

In [ ]:
for rd in "ba":
    print(f"chirp reading {rd}:", {sb: mwu(hc.loc[(hc.prep == "culture") & (hc["sub"] == sb), f"f3db_{rd}"],
                                         hc.loc[(hc.prep == "acute") & (hc["sub"] == sb), f"f3db_{rd}"])["p"] for sb in SUBS})
print("Table S3: LAMP5 p = 0.166, VIP 0.046, PV 0.031, SST 0.112")

**Result: not reproduced.** f3dB differs between cultured and acute cells in PV (p = 0.014) and SST (p = 0.002) under chirp reading (b), and also in VIP (p = 0.040) under reading (a). The paper's own Table S3 gives p < 0.05 for VIP and PV.

## Patch-seq expression

The expression of the human cells comes from the analysis repositories of Chartrand et al.
(2023) (CPM, layer 1 cells) and Lee, Dalley et al. (2023) (log2(CPM + 1), converted to CPM),
matched to the cells by specimen id. Cells in both matrices take the Chartrand values; how the
two agree is printed. Mouse expression (Gouwens et al. 2020) is not read, so the mouse part of
figs4 is not tested.

In [ ]:
GENES6B = ["HCN1", "KCNQ3", "KCNQ5", "CACNA1A", "CACNA1G", "KCNJ16"]
FAMILY = ["KCNJ16", "HCN1", "CACNA1A", "KCNQ3", "SCN8A", "KCNQ5", "KCNH7", "SCN1A", "KCNQ2", "CACNA1G"]
expr, expr_sums = R.expression(sorted(set(GENES6B + FAMILY)))
print("column sums of CPM per matrix (should be 1e6):", {k: (round(float(v.min())), round(float(v.max()))) for k, v in expr_sums.items()})
both = expr.index[expr.index.duplicated(keep=False)].unique()
if len(both):
    x = expr.loc[both]
    c1, c2 = x[x.expr_source == "chartrand"].groupby(level=0).first(), x[x.expr_source != "chartrand"].groupby(level=0).first()
    print(f"cells in both matrices: {len(both)}; Pearson r of log CPM between them, HCN1 {np.corrcoef(np.log1p(c1['HCN1']), np.log1p(c2.loc[c1.index, 'HCN1']))[0, 1]:.3f}")
expr = expr[~expr.index.duplicated(keep="first")]
hx = hum[hum["sub"].notna()].merge(expr, left_on="spec", right_index=True, how="inner")
print("human cells with expression:", hx.groupby(["prep", "sub"]).size().to_dict(), "| with a usable chirp:",
      hx[hx.chirp_ok].groupby(["prep", "sub"]).size().to_dict())
print("paper: acute LAMP5 256, VIP 116, PV 121, SST 20 (Fig. 6b); cultured LAMP5 31, VIP 46, PV 271, SST 148 (Fig. S4a)")

## fig6b-channel-expression

> In acute and cultured human slices, all interneuron subclasses express high levels of HCN1,
> KCNQ3/5 and CACNA1A/G transcripts.

Per gene and subclass, the fraction of cells with CPM > 1 and the mean CPM. Reading (a): more
than half the cells express each gene in every subclass; reading (b): each gene's mean CPM
exceeds that of KCNJ16 in every subclass. Reproduced when a reading holds in both preparations.

In [ ]:
for pr in ("acute", "culture"):
    g = hx[hx.prep == pr]
    frac = g.groupby("sub")[GENES6B].agg(lambda v: (v > 1).mean()).round(2)
    mean = g.groupby("sub")[GENES6B].mean().round(1)
    print(f"{pr}: cells {g['sub'].value_counts().to_dict()}")
    print("  fraction with CPM > 1:"); print(frac.to_string())
    print("  mean CPM:"); print(mean.to_string())
    ra = bool((frac[GENES6B[:5]] > 0.5).all().all())
    rb = bool(mean[GENES6B[:5]].gt(mean["KCNJ16"], axis=0).all().all())
    print(f"  reading a holds: {ra}; reading b holds: {rb}")
    print("  genes failing reading a:", {sb: [k for k in GENES6B[:5] if frac.loc[sb, k] <= 0.5] for sb in frac.index},
          "| failing reading b:", {sb: [k for k in GENES6B[:5] if mean.loc[sb, k] <= mean.loc[sb, "KCNJ16"]] for sb in mean.index})

**Result: reproduced.** In acute cells (LAMP5 208, PV 115, VIP 82, SST 19 with expression) and cultured cells (PV 251, SST 139, VIP 42, LAMP5 24), HCN1, KCNQ3, KCNQ5, CACNA1A and CACNA1G are expressed (CPM > 1) by more than half the cells of every subclass (lowest fraction 0.67, cultured LAMP5 KCNQ5 and CACNA1G), and each has a higher mean CPM than KCNJ16 in every subclass (KCNJ16 3.8 to 9.7 CPM; lowest of the five, CACNA1G 33.4). Both readings hold in both preparations. CACNA1G, which the paper also lists among the lower genes, has the lowest mean of the five.

## fig6c-gene-resonance-correlation

> In acute human interneurons, KCNJ16 expression correlates positively with resonance strength
> and frequency, and HCN1 and CACNA1A expression correlate positively with resonance frequency,
> after FDR correction.

Pearson correlation of expression with Q and fres over acute cells with a chirp and CPM > 1 for
the gene, BH-FDR over the 20 tests of the acute family (10 genes by 2 measures), under both
readings of the scale (O8: CPM, log(CPM + 1)) and both chirp readings. Reproduced when the four
correlations (KCNJ16 with Q and fres, HCN1 and CACNA1A with fres) are positive with adjusted p <
0.05 and within tolerance (0.28, 0.20, 0.17, 0.16, each +/- 10%); partially reproduced when three are.

In [ ]:
from statsmodels.stats.multitest import multipletests
def gene_corr(d, rd, scale):
    rows = []
    for gname in FAMILY:
        x = d[d[gname] > 1]
        e = x[gname] if scale == "cpm" else np.log1p(x[gname])
        for v in ("Q", "fres"):
            r, p = stats.pearsonr(e, x[f"{v}_{rd}"])
            rows.append(dict(gene=gname, measure=v, n=len(x), r=round(r, 3), p=p))
    t = pd.DataFrame(rows)
    t["p_fdr"] = multipletests(t["p"], method="fdr_bh")[1]
    return t
ha6 = hx[(hx.prep == "acute") & hx.chirp_ok]
key = [("KCNJ16", "Q"), ("KCNJ16", "fres"), ("HCN1", "fres"), ("CACNA1A", "fres")]
for rd in "ba":
    for scale in ("cpm", "log"):
        t = gene_corr(ha6, rd, scale)
        k = t.set_index(["gene", "measure"]).loc[key]
        print(f"chirp reading {rd}, scale {scale}: the four tested correlations")
        print(k[["n", "r", "p", "p_fdr"]].round(4).to_string())
        print("   other correlations with FDR p < 0.05:", t[(t.p_fdr < 0.05)][["gene", "measure", "r"]].values.tolist())
print("paper: KCNJ16-Q r = 0.279 (n 204), KCNJ16-fres 0.202 (204), HCN1-fres 0.174 (240), CACNA1A-fres 0.164 (254)")

**Result: not reproduced.** Over acute cells with a chirp (chirp reading b, CPM), HCN1 correlates with fres (r = 0.326, n = 189, FDR p = 1e-4; outside 0.17 +/- 0.017), but KCNJ16 does not correlate with Q (r = -0.023, n = 165, FDR p = 0.94) or fres (r = -0.042), and CACNA1A with fres misses after correction (r = 0.140, FDR p = 0.24). One of the four holds, under every reading of the scale and chirp (HCN1 r = 0.28 to 0.33; KCNJ16 |r| <= 0.07; CACNA1A FDR p >= 0.24).

## figs4-culture-mouse-genes

> No gene-resonance correlation survives correction in cultured human interneurons; among mouse
> interneurons only HCN2 is significantly related to resonance.

The cultured part as in fig6c, on cultured cells (a claim of absence at FDR p < 0.05). The mouse
expression is not read here, so the mouse part is not tested.

In [ ]:
hc6 = hx[(hx.prep == "culture") & hx.chirp_ok]
for rd in "ba":
    for scale in ("cpm", "log"):
        t = gene_corr(hc6, rd, scale)
        print(f"chirp reading {rd}, scale {scale}: cultured cells {len(hc6)}; smallest FDR p {t.p_fdr.min():.3g};",
              "FDR p < 0.05:", t[t.p_fdr < 0.05][["gene", "measure", "n", "r"]].values.tolist())

**Result: not reproduced.** In the 181 cultured cells with a chirp, several correlations survive FDR correction under every reading, among them HCN1 with Q (r = 0.262) and fres (r = 0.317) and CACNA1A with fres (r = 0.201) (chirp reading b, CPM; smallest FDR p = 5e-4), where the paper finds none. The mouse part (HCN2) is not tested, as the mouse expression is not read here.

## figs1-region-pathology

> Resonance metrics (fres, Q, 3 dB cutoff) and input resistance do not differ between brain
> regions of resection or between epilepsy and tumour cases; sag ratio does.

Lobe and disease per donor from Table S1: its "Lobe" column where it is filled, otherwise its
"Area" grouped into lobes (MTG, STG, ITG, TemL temporal; SFG, MFG, IFGtr, PRG, LOrG, PaCG,
FroL frontal; SPL, AnG, ParL parietal; IOG, SOG, OccL occipital, a standard choice); Table S1
has a "Structure" column too, which was not read. Disease is epilepsy (or epileptiform)
against tumour. Kruskal-Wallis across lobes and two-sided Mann-Whitney U between epilepsy and
tumour, on acute cells (criterion reading a) and on all human cells (reading b). Reproduced
when fres, Q, f3dB and input resistance give p >= 0.05 in both tests.

In [ ]:
area_lobe = {"MTG": "temporal", "STG": "temporal", "ITG": "temporal", "TemL": "temporal", "PLP": "temporal",
             "SFG": "frontal", "MFG": "frontal", "IFGtr": "frontal", "PRG": "frontal", "LOrG": "frontal",
             "PaCG": "frontal", "FroL": "frontal", "SPL": "parietal", "AnG": "parietal", "ParL": "parietal",
             "IOG": "occipital", "SOG": "occipital", "OccL": "occipital"}
lobe_col = {"TemL": "temporal", "FroL": "frontal", "ParL": "parietal", "OccL": "occipital"}
def s1_lobe(d):
    ds, area, dis, lobe = R.TABLE_S1.get(d, ("", "", "", ""))
    return lobe_col.get(lobe) or area_lobe.get(area)
def disease(d):
    x = R.TABLE_S1.get(d, ("", "", "", ""))[2].lower()
    return "tumour" if "tumor" in x else ("epilepsy" if "epilep" in x else None)
hum = hum.assign(lobe=hum["donor"].map(s1_lobe), disease=hum["donor"].map(disease))
print("human donors not in Table S1:", sorted(set(hum["donor"].dropna()) - set(R.TABLE_S1)))
def region_tests(d, measures):
    rows = []
    for v in measures:
        x = d.dropna(subset=[v])
        x = x[x[v] > 0] if v.startswith("rin") else x
        l = x.dropna(subset=["lobe"])
        kw = stats.kruskal(*[g[v].values for _, g in l.groupby("lobe")])
        e, t = x.loc[x["disease"] == "epilepsy", v], x.loc[x["disease"] == "tumour", v]
        mw = stats.mannwhitneyu(e, t, alternative="two-sided")
        rows.append(dict(measure=v, KW=round(kw.statistic, 2), KW_p=float(f"{kw.pvalue:.3g}"), MWU=mw.statistic, MWU_p=float(f"{mw.pvalue:.3g}"),
                         n_epi=len(e), n_tum=len(t), med_epilepsy=round(e.median(), 3), med_tumour=round(t.median(), 3),
                         **{f"n_{k}": len(g) for k, g in l.groupby("lobe")}))
    return pd.DataFrame(rows)
for nm, d in [("acute cells (reading a)", hum[hum.prep == "acute"]), ("all human cells (reading b)", hum)]:
    print(f"\n{nm}: lobes {d.loc[d.chirp_ok, 'lobe'].value_counts(dropna=False).to_dict()} (chirp cells), donors per lobe "
          f"{d.groupby('lobe')['donor'].nunique().to_dict()}; disease {d.loc[d.chirp_ok, 'disease'].value_counts(dropna=False).to_dict()}")
    print("chirp measures over chirp cells; sag and input resistance over all cells with a sag ratio")
    print(pd.concat([region_tests(d[d.chirp_ok], ["fres_b", "Q_b", "f3db_b", "fres_a", "Q_a", "f3db_a"]),
                     region_tests(d[d.sag_ok], ["rin_a", "rin_b", "sag_a", "sag_b"])]).to_string())
# Which cells the paper used: under the null a Mann-Whitney U is near n1 * n2 / 2. The paper's
# input resistance test (U = 8207, p = 0.78) implies n1 * n2 near 16400.
for nm, d in [("acute", hum[(hum.prep == "acute") & hum.sag_ok]), ("all human", hum[hum.sag_ok])]:
    print(f"n_epilepsy x n_tumour for input resistance, {nm} cells here:",
          int((d["disease"] == "epilepsy").sum() * (d["disease"] == "tumour").sum()), "(paper about 2 x 8207 = 16414)")

**Result: reproduced, under the assumed readings (acute cells, chirp reading b).** On human acute cells, fres, Q, f3dB and input resistance differ neither between lobes (Kruskal-Wallis p = 0.92, 0.51, 0.38, 0.34) nor between epilepsy and tumour cases (Mann-Whitney p = 0.54, 0.052, 0.88, 0.31). Q is near the threshold: under chirp reading (a) it differs between epilepsy and tumour (p = 0.043). Sag differs between lobes (p = 0.012) and is lower in epilepsy (0.117 against 0.162, p = 0.001), as the paper reports. On all human cells (criterion reading b) fres and Q differ by lobe and disease (p = 0.008 to 0.048), and input resistance by lobe (p = 0.005). The paper's input resistance test (U = 8207 at p = 0.78) implies about 16400 epilepsy-tumour pairs, nearer the acute cells (29192) than all cells (157680), which supports reading (a). The occipital lobe has 1 acute cell.

### Variant: the 000630 cells alone

The round-1 analysis used the 000630 cells only (LAMP5 and VIP). The main readings on those
cells, for comparison with the four-subclass results above:

In [ ]:
for nm, meas in [("fig3eg fres", "fres"), ("fig3ik Q", "Q")]:
    t = fig3_fits(acute[acute.source == "000630"], meas, reduced=(meas == "Q")).query("chirp == 'b' and sag_rin == 'a'")
    print(nm); show_fit(t, ["reduced", "full", "LAMP5", "VIP"] if meas == "Q" else ["full", "LAMP5", "VIP"])

### Variant: the LOWESS fraction, carried to the outcomes

The claims file's O1 fixes the LOWESS fraction at 0.04 for both chirp readings; the smoothing
table above shows that it moves the resonant fractions. Here the criteria of the claims that rest
on fres or the resonant fraction are recomputed at 0.02 and 0.08 (chirp reading b, other choices
as in the main analysis).

In [ ]:
# fres, Q and f3dB under each LOWESS fraction (chirp reading b), for every cell with a usable chirp
def smoothed(frac):
    vals = smooth_vals[frac]
    d = allc[allc["chirp_ok"]].copy()
    for v in ("fres", "Q"):
        d[f"{v}_s"] = [vals[(s_, x)][v] for s_, x in zip(d["source"], d["session_id"])]
    return d
def pct(x):
    return f"{int(x.sum())}/{len(x)} = {100 * x.mean():.1f}%"
lowess_rows = []
for frac in (0.02, 0.04, 0.08):
    d = smoothed(frac)
    d["res"] = R.is_resonant(d["fres_s"], d["Q_s"])
    h, mo_, cu = d[(d.species == "human") & (d.prep == "acute")], d[d.species == "mouse"], d[(d.species == "human") & (d.prep == "culture") & d["sub"].notna()]
    print(f"--- LOWESS fraction {frac}")
    kh, nh, km, nm_ = int(h.res.sum()), len(h), int(mo_.res.sum()), len(mo_)
    print(f"  fig2c: human {pct(h.res)} (32.7 +/- 3.27), mouse {pct(mo_.res)} (9.3 +/- 0.93); "
          f"gof p = {R.chi2_gof(kh, nh, km / nm_)[1]:.3g}, 2x2 p = {R.chi2_2x2(kh, nh, km, nm_)[1]:.3g}")
    out = []
    for sb in ("LAMP5", "VIP"):
        hs, ms = h[h["sub"] == sb], mo_[mo_["sub"] == sb]
        out.append(f"{sb} human {pct(hs.res)}, mouse {pct(ms.res)}, p = {R.chi2_gof(int(hs.res.sum()), len(hs), ms.res.mean())[1]:.3g}")
    print("  fig2d-subclass:", "; ".join(out), f"; PV human {pct(h[h['sub'] == 'PV'].res)} (LAMP5 31.25, VIP 25.0, PV 43.6, +/- 10%)")
    kc, nc, ka, na = int(cu.res.sum()), len(cu), int(h.res.sum()), len(h)
    out = []
    for sb in SUBS:
        c_, a_ = cu[cu["sub"] == sb], h[h["sub"] == sb]
        out.append(f"{sb} {pct(c_.res)} against {100 * a_.res.mean():.1f}%, p = {R.chi2_gof(int(c_.res.sum()), len(c_), a_.res.mean())[1]:.3g}")
    print(f"  fig5i: cultured {pct(cu.res)} (60.8 +/- 6.08), acute {100 * ka / na:.1f}%; gof p = {R.chi2_gof(kc, nc, ka / na)[1]:.3g};", "; ".join(out))
    lv = d[(d.prep == "acute") & d["sub"].isin(["LAMP5", "VIP"])]
    lv = pd.DataFrame(dict(y=lv["fres_s"], species=lv["species"], sub=lv["sub"], vm=lv["vm_a"], g=lv["species"] + ":" + lv["donor"].astype(str)))
    form = "y ~ C(species, Sum) * C(sub, Sum) + vm"
    r = mixed(form, lv, "g", {"interaction": [c for c in cols_of(lv, form) if ":" in c]})
    mw = {sb: mwu(lv.loc[(lv.species == "human") & (lv["sub"] == sb), "y"], lv.loc[(lv.species == "mouse") & (lv["sub"] == sb), "y"]) for sb in ("LAMP5", "VIP")}
    print(f"  fig2e: interaction p = {r['interaction a'][2]} (conv {r['conv_a']}); VIP human against mouse p = {mw['VIP']['p']} "
          f"(medians {mw['VIP']['med_x']}, {mw['VIP']['med_y']}); fig2e-lamp5: LAMP5 p = {mw['LAMP5']['p']} (medians {mw['LAMP5']['med_x']}, {mw['LAMP5']['med_y']})")
    f3 = fig3_frame(h.assign(fres_b=h["fres_s"]), "fres", "b", "a")
    form = "y ~ sag * C(sub, Sum) + rin * C(sub, Sum)"
    cs = cols_of(f3, form)
    r = mixed(form, f3, "g", {"sag x sub": [c for c in cs if c.startswith("sag:")], "rin x sub": [c for c in cs if c.startswith("rin:")]})
    l5 = mixed("y ~ sag + rin", f3[f3["sub"] == "LAMP5"], "g", {"sag": ["sag"], "rin": ["rin"]})
    print(f"  fig3eg: sag x subclass p = {r['sag x sub a'][2]}, input resistance x subclass p = {r['rin x sub a'][2]}; "
          f"LAMP5 sag {l5['sag a'][0]} (p = {l5['sag a'][2]}), input resistance p = {l5['rin a'][2]}")


The outcomes of four claims rest on the LOWESS fraction that O1 fixes at 0.04. At 0.02,
fig2c's fractions are both within tolerance (34.4% and 9.4%, p = 4e-36 and 9e-8), which would
reproduce it; fig2d-subclass gives LAMP5 31.0% (within tolerance, p = 2.3e-9), VIP 20.4% (below
tolerance) and PV 49.2% (above); fig5i gives a cultured fraction of 69.1% (above tolerance) and
SST p = 0.22. At 0.08 the human acute fractions fall far below the paper's (9.9% overall, LAMP5
3.4%), and the fig2e-lamp5 LAMP5 cells differ in fres (p = 0.003, against 0.097 at 0.02 and 0.24
at 0.04). The fig2e interaction holds at every fraction (p = 0.013, 0.004, 0.011), and the fig3eg
interactions miss at every fraction (p >= 0.10). No outcome is changed for this, as the claims
file fixes the fraction; the notes of these claims say they rest on it.

### Variant: fres at the 0.4 Hz floor

Under chirp reading (b) a cell whose impedance peaks at or below 0.4 Hz gets fres = 0.4 Hz, a
censored value. The paper's Table S3 medians of 0.40 Hz show that its data have the same floor,
and its models treat fres as a linear outcome, as the main analysis does. Two variants show how
much the fres models of fig2e, fig3eg and fig3fh rest on that.

In [ ]:
# fres at the 0.4 Hz floor (chirp reading b) is a censored value: the peak lies at or below the
# lowest frequency kept. Two variants: fres replaced by its rank-based inverse normal score
# (cells at the floor tie at the lowest rank), and cells at the floor left out.
def rin_score(y):
    return pd.Series(stats.norm.ppf((y.rank() - 0.5) / len(y)), index=y.index)
FLOOR = 0.41
lv = species_frame("fres", "b", "a")
lv = lv[lv["sub"].isin(["LAMP5", "VIP"])][["y", "species", "sub", "vm", "g"]].dropna()
print("fres at the floor (<= 0.41 Hz), LAMP5 and VIP cells:", lv.assign(fl=lv.y <= FLOOR).groupby(["species", "sub"])["fl"].agg(["sum", "count"]).to_dict("index"))
form = "y ~ C(species, Sum) * C(sub, Sum) + vm"
for nm, d in [("rank-based inverse normal", lv.assign(y=rin_score(lv["y"]))), ("floor cells left out", lv[lv.y > FLOOR])]:
    cs = cols_of(d, form)
    r = mixed(form, d, "g", {"interaction": [c for c in cs if ":" in c], "species": [c for c in cs if c.startswith("C(species") and ":" not in c]})
    print(f"fig2e, {nm}: n = {r['n']}, interaction {r['interaction a']}, species {r['species a']}, converged {r['conv_a']}")
for nm0, d0, subs in [("fig3eg, human acute", acute, ["LAMP5", "VIP", "PV", "SST"]), ("fig3fh, mouse", mouse, ["LAMP5", "VIP"])]:
    base = fig3_frame(d0, "fres", "b", "a")
    print(f"{nm0}: cells at the floor {int((base.y <= FLOOR).sum())} of {len(base)}")
    for nm, d in [("rank-based inverse normal", base.assign(y=rin_score(base["y"]))), ("floor cells left out", base[base.y > FLOOR])]:
        form = "y ~ sag * C(sub, Sum) + rin * C(sub, Sum)"
        cs = cols_of(d, form)
        r = mixed(form, d, "g", {"sag": ["sag"], "rin": ["rin"], "sag x sub": [c for c in cs if c.startswith("sag:")], "rin x sub": [c for c in cs if c.startswith("rin:")]})
        per = {sb: {k: mixed("y ~ sag + rin", gg, "g", {"sag": ["sag"], "rin": ["rin"]})[f"{k} a"][2] for k in ("sag", "rin")} | {"n": len(gg)}
               for sb, gg in d.groupby("sub") if sb in subs and len(gg) > 5}
        print(f"  {nm}: n = {r['n']}; sag p = {r['sag a'][2]}, input resistance p = {r['rin a'][2]}, sag x subclass p = {r['sag x sub a'][2]}, "
              f"input resistance x subclass p = {r['rin x sub a'][2]}, converged {r['conv_a']}")
        print("     per subclass p:", per)


With fres replaced by rank-based inverse normal scores, or with the cells at the floor left
out, the fig2e species by subclass interaction is not significant (p = 0.31 and 0.46). The fig2e
outcome therefore rests on treating floored fres as a linear outcome, as the paper's model does.
For fig3eg the interactions stay non-significant (sag p = 0.80 and 0.18, input resistance p =
0.16 and 0.19) and LAMP5 input resistance gives p = 0.21 and 0.60, so its outcome is the same.
For fig3fh input resistance gives p = 0.055 and 0.56, so it still does not predict fres at
p < 0.05.

### Variant: cultured cells without donor H19.03.002

20 cultured cells of 000636 come from donor H19.03.002, whom Table S1 does not list
(lee-label-link), so they may not be in the paper's sample. The culture claims without them:

In [ ]:
# Cultured cells of donor H19.03.002, who is not in Table S1, left out
hc = hum[hum["chirp_ok"] & hum["sub"].notna()]
hs = hum[hum["sag_ok"] & hum["sub"].notna()]
print("cultured cells of H19.03.002:", int((cult["donor"] == "H19.03.002").sum()), "| with a chirp:",
      int(((cult["donor"] == "H19.03.002") & cult["chirp_ok"]).sum()), "| by subclass:", cult.loc[cult["donor"] == "H19.03.002", "sub"].value_counts().to_dict())
hcx, hsx = hc[hc["donor"] != "H19.03.002"], hs[hs["donor"] != "H19.03.002"]
sel = lambda d, pr, sb: d[(d.prep == pr) & (d["sub"] == sb)]
print("fig5g sag (reading a), cultured against acute:", {sb: (mwu(sel(hsx, "culture", sb)["sag_a"], sel(hsx, "acute", sb)["sag_a"])["p"],
      mwu(sel(hsx, "culture", sb)["sag_a"], sel(hsx, "acute", sb)["sag_a"])["med_x"]) for sb in ("LAMP5", "PV")})
d = hsx[hsx["rin_a"] > 0]
d = pd.DataFrame(dict(y=d["rin_a"], prep=d["prep"], sub=d["sub"], vm=d["vm_b"], g=d["donor"].astype(str))).dropna()
form = "y ~ C(prep, Sum) * C(sub, Sum) + vm"
r = mixed(form, d, "g", {"prep": [c for c in cols_of(d, form) if c.startswith("C(prep") and ":" not in c]})
print("fig5g input resistance (reading a), preparation effect:", r["prep a"], "n =", r["n"])
r_ = R.is_resonant(hcx["fres_b"], hcx["Q_b"])
cu, ac_ = r_[hcx.prep == "culture"], r_[hcx.prep == "acute"]
print(f"fig5i (chirp reading b): cultured {int(cu.sum())}/{len(cu)} = {100 * cu.mean():.1f}%, gof p = {R.chi2_gof(int(cu.sum()), len(cu), ac_.mean())[1]:.3g};",
      {sb: f"{100 * R.is_resonant(sel(hcx, 'culture', sb)['fres_b'], sel(hcx, 'culture', sb)['Q_b']).mean():.1f}%, p = "
           f"{R.chi2_gof(int(R.is_resonant(sel(hcx, 'culture', sb)['fres_b'], sel(hcx, 'culture', sb)['Q_b']).sum()), len(sel(hcx, 'culture', sb)), R.is_resonant(sel(hcx, 'acute', sb)['fres_b'], sel(hcx, 'acute', sb)['Q_b']).mean())[1]:.3g}"
       for sb in SUBS})
print("fig5jk (chirp reading b), p and cultured median:", {f"{v} {sb}": (mwu(sel(hcx, "culture", sb)[f"{v}_b"], sel(hcx, "acute", sb)[f"{v}_b"])["p"],
      mwu(sel(hcx, "culture", sb)[f"{v}_b"], sel(hcx, "acute", sb)[f"{v}_b"])["med_x"]) for v, sb in [("fres", "LAMP5"), ("fres", "PV"), ("Q", "LAMP5"), ("Q", "PV"), ("Q", "SST")]})
print("fig5l f3dB (chirp reading b), p:", {sb: mwu(sel(hcx, "culture", sb)["f3db_b"], sel(hcx, "acute", sb)["f3db_b"])["p"] for sb in SUBS})
hc6x = hx[(hx.prep == "culture") & hx.chirp_ok & (hx["donor"] != "H19.03.002")]
t = gene_corr(hc6x, "b", "cpm")
print(f"figs4 (chirp reading b, CPM): cultured cells {len(hc6x)}; FDR p < 0.05:", t[t.p_fdr < 0.05][["gene", "measure", "n", "r"]].values.tolist())


Without the 20 cells of H19.03.002 every culture result keeps its outcome. fig5g: cultured
sag is higher in LAMP5 (p = 7.4e-7, median 0.260) and PV (p = 1.2e-34, median 0.282), and the
preparation effect on input resistance stays significant (p = 0.0036). fig5i: 61.7% of cultured
cells are resonant (within tolerance) and SST still misses (p = 0.087). fig5jk: the same four
comparisons hold (SST Q p = 0.34), and the cultured LAMP5 fres (2.75 Hz) and PV Q (1.190) medians
stay outside tolerance. fig5l: f3dB still differs in PV (p = 0.016) and SST (p = 0.0018). figs4:
four correlations still survive FDR correction.

## Claims on cortical depth

fig4c, fig4df, fig4g and figs2-l23 need the pia-soma distance of human cells across layers;
fig4i and fig4km need it for mouse cells. The Lee, Dalley et al. table has no column naming a
depth or distance (columns printed in the data check). The Chartrand table's
`Depth_from_Pia_microns` is 0 for every cell; its `soma_depth_um` is filled for a few cells
(printed below). The Allen mouse metadata has an empty `depth_from_pia_um` column and a
normalized depth for few cells.

In [ ]:
print("human acute cells by layer label (000630 from the Chartrand table's target_layer, 000636 from the Lee table):",
      acute.groupby("source")["layer"].value_counts().to_dict())
print("human acute cells in layers 2 and 3:", int(acute["layer"].isin(["2", "3"]).sum()))
sd = ct.loc[ct["soma_depth_um"].notna(), ["spec_id", "soma_depth_um"]].assign(spec=lambda x: pd.to_numeric(x["spec_id"], errors="coerce").astype("Int64").astype(str))
hd = acute.merge(sd[["spec", "soma_depth_um"]], on="spec", how="inner")
print("Chartrand table rows with soma_depth_um:", len(sd), f"({sd['soma_depth_um'].min():.0f} to {sd['soma_depth_um'].max():.0f} um)",
      "| human acute cells here with it:", len(hd), hd.groupby(["source", "layer", "sub"]).size().to_dict(),
      "| with a sag ratio:", int(hd["sag_ok"].sum()), "| with a chirp:", int(hd["chirp_ok"].sum()))
print("000630 cells with a pia-soma distance:", int(acute["Depth_from_Pia_microns"].notna().sum()),
      f"(range {acute['Depth_from_Pia_microns'].min():.0f} to {acute['Depth_from_Pia_microns'].max():.0f} um); paper's human range 22 to 2379 um")
print("mouse cells with a normalized soma depth:", int(mouse["cell_soma_normalized_depth"].notna().sum()), "of", len(mouse),
      "| with a chirp:", int((mouse["cell_soma_normalized_depth"].notna() & mouse["chirp_ok"]).sum()))

## Summary

| Claim | Outcome |
|---|---|
| chartrand-label-link | reproduced |
| lee-label-link | reproduced |
| mouse-specimen-link | partially reproduced (461 of 628 Table S2b cells public; no PV or SST) |
| sample-size | not reproduced (440 human acute cells from 76 donors) |
| fig2b-human-higher-fres-q | reproduced (under both chirp readings) |
| fig2ab-peak-range | not reproduced (mouse median 1.70 Hz) |
| fig2c-resonant-proportion | not reproduced (assumed chirp reading b; reading a reproduces) |
| fig2d-subclass-proportion | not reproduced |
| fig2d-pv-highest | reproduced |
| figs2a-stringent-criteria | reproduced |
| fig2e-fres-species-subclass | partially reproduced (assumed chirp reading b) |
| fig2e-lamp5-sst-fres-similar | partially reproduced (assumed chirp reading b; SST not testable) |
| fig2f-f3db-species-subclass | partially reproduced |
| fig2g-q-species-subclass | not reproduced |
| fig3c-sag-species | partially reproduced |
| fig3d-input-resistance-species | partially reproduced |
| fig3eg-human-fres-sag-rin | not reproduced |
| fig3ik-human-q-sag-rin | partially reproduced (assumed readings) |
| fig3fh-mouse-fres | not reproduced |
| fig3jl-mouse-q-sag-rin | reproduced |
| fig5g-culture-sag | partially reproduced |
| fig5i-culture-proportion | partially reproduced (assumed chirp reading b; reading a reproduces) |
| fig5jk-culture-fres-q | partially reproduced |
| fig5l-culture-f3db-same | not reproduced |
| fig6b-channel-expression | reproduced |
| fig6c-gene-resonance-correlation | not reproduced |
| figs4-culture-mouse-genes | not reproduced (cultured part; mouse part not tested) |
| figs1-region-pathology | reproduced (assumed readings) |
| the other 19 claims | blocked (see the attempt record) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The human acute sample is 440 cells (212 with a usable
chirp) where the paper's is 542 (297); the mouse sample is the 461 Table S2b cells in 000020
(139 with a chirp), with no PV or SST cells; the cultured sample is the 456 cells of 000636 (181
with a chirp) where the paper's is 543 (268). The paper's chirp, sag and input resistance code
is not public, so the IPFX v2.0.0 computations are used, with the readings the claims file
names. The chirp profile is kept from 0.4 Hz (reading b of O1), which matches Table S3's medians
of 0.40 Hz, the mouse LAMP5 medians, and the Allen Institute's own chirp features; reading (a),
from 0.2 Hz, gives resonant fractions nearer the paper's and is shown throughout. The LOWESS
fraction (0.04, the claims file's) changes the human fractions and medians strongly (smoothing
table), and the outcomes of fig2c, fig2d-subclass-proportion, fig5i and fig2e-lamp5 rest on it
(LOWESS variant: at 0.02 fig2c would be reproduced). fres at the 0.4 Hz floor is treated as a
linear outcome, as in the paper's models; the fig2e interaction does not survive a rank-based
or floor-excluding variant. Leaving out the 20 cultured cells of donor H19.03.002, who is not in
Table S1, changes no culture result. Mixed models use Satterthwaite tests computed here (O5 reading a), with
statsmodels Wald tests shown.

What was found about the datasets. Every 000630 file links to the Chartrand et al. table, and 690
of the 706 000636 files to the Lee, Dalley et al. table, by session id. Four cells are deposited in
both dandisets, with identical measures (fres, Q, sag ratio and input resistance) in both copies. 461 of the paper's 628 mouse cells are in 000020; the files
carry no specimen id, so they are found only through the Allen metadata, which the Allen portal
gives at no fixed address. The files' subject records hold only the subject id, age, sex, species, genotype and a
description (and date of birth in 000020), so subclass, layer and depth come from the label
tables; the Chartrand table's pia-soma distance is 0 for every 000630 cell, and its
soma_depth_um is filled for 22 rows (20 cells here). The Chartrand table puts 38 of the 210
cells of 000630, which the dandiset describes as layer 1 neurons, in layers 2 to 4. In 000636,
21 files from 6 subjects give the donor's age as P0.0D. Many
sweeps stop early and are zero-filled without a flag.